# Modelo de difusão com mudança de regime por limiares lineares afins

O preço $X_t$ segue um movimento browniano geométrico
$dX_t = \mu_r X_t\,dt + \sigma_r X_t\,dB_t$, cujos parâmetros dependem do regime
$r \in \{1, 2\}$. A mudança de regime é autoinduzida por duas fronteiras lineares:

- $M(t) = a_1 t + b_1$: no regime 1, tocar $M(t)$ faz passar ao regime 2;
- $m(t) = a_2 t + b_2$: no regime 2, tocar $m(t)$ faz passar ao regime 1.

O vetor de limiares é $\theta = (a_1, b_1, a_2, b_2)$ e o modelo só está definido
enquanto $m(t) < M(t)$.

**Estrutura do notebook**

1. Simulação do processo
2. Classificação em regimes e estimação por máxima verosimilhança
3. Geração de sementes para a otimização
4. Estimação dos limiares afins (Nelder–Mead multi-start)
5. Modelos de referência: MBG e limiares constantes
6. Comparação de modelos (AIC/BIC)
7. Diagnóstico dos parâmetros por regime
8. Identificabilidade: mapas de verosimilhança, refinamento e análise de focos
9. Validação fora da amostra
10. Gráfico dos regimes
11. Pipeline completo e aplicação a um conjunto de ativos

**Convenções:** tempo em anos, com $\Delta t = 1/252$ (dias úteis); $\mu$ e $\sigma$
anualizados.

In [67]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import stats
from scipy.ndimage import label
from scipy.optimize import minimize
from scipy.stats import chi2
from sklearn.linear_model import LinearRegression

%matplotlib inline

DIAS_UTEIS_ANO = 252

## 1. Simulação do processo

In [68]:
def simular_processo(T, dt, x0, params_regime1, params_regime2, theta, regime_inicial=1):
    '''Simula uma trajetória do modelo com mudança de regime por limiares afins.

    Em cada passo, o regime é atualizado com base no preço e nos limiares do 
    instante anterior(versão discreta da regra de comutação, adequada para dt
    pequeno) e o preço evolui pela solução exata do MBG nesse regime.
    O ponto que dispara a mudança (x[i-1], já para lá da fronteira)
    foi gerado com os parâmetros do regime cessante. É o ponto seguinte, x[i],
    que é rotulado em regimes[i] e gerado já com os parâmetros do regime novo.

    Parâmetros
    ----------
    T : float
        Horizonte da simulação, em anos.
    dt : float
        Passo temporal, em anos.
    x0 : float
        Preço inicial.
    params_regime1, params_regime2 : tuple (mu, sigma)
        Drift e volatilidade anualizados de cada regime.
    theta : sequência (a1, b1, a2, b2)
        Declives e ordenadas na origem de M(t) e m(t).
    regime_inicial : int
        Regime em t = 0.

    Devolve
    -------
    (t, x, regimes) : arrays com os instantes, os preços e o regime em vigor
    em cada instante.
    '''
    n = int(T / dt)
    t = np.arange(n) * dt
    x = np.zeros(n)
    regimes = np.zeros(n, dtype=int)
    x[0] = x0
    regime = regime_inicial
    regimes[0] = regime
    parametros = {1: params_regime1, 2: params_regime2}
    a1, b1, a2, b2 = theta
    for i in range(1, n):
        L_M = a1 * t[i-1] + b1
        L_m = a2 * t[i-1] + b2
        if regime == 1 and x[i-1] >= L_M:
            regime = 2
        elif regime == 2 and x[i-1] <= L_m:
            regime = 1
        regimes[i] = regime
        mu, sigma = parametros[regime]
        dB = np.random.normal(0, np.sqrt(dt))
        x[i] = x[i-1] * np.exp((mu - 0.5 * sigma**2) * dt + sigma * dB)
    return t, x, regimes

## 2. Classificação em regimes e estimação por máxima verosimilhança

In [69]:
def _classificar_regimes(t, x, theta, regime_inicial=1):
    '''Atribui a cada instante o regime em vigor, dados os limiares theta.

    Aplica a mesma regra de comutação do simulador: no regime 1 passa-se ao
    regime 2 quando x[i-1] >= M(t[i-1]); no regime 2 passa-se ao regime 1
    quando x[i-1] <= m(t[i-1]).

    Parâmetros
    ----------
    t, x : array
        Instantes (anos) e preços observados.
    theta : sequência (a1, b1, a2, b2)
        Declives e ordenadas na origem de M(t) e m(t).
    regime_inicial : int
        Regime assumido em t[0].

    Devolve
    -------
    regimes : array de inteiros em {1, 2}, um por instante de t.
    '''
    a1, b1, a2, b2 = theta
    n = len(x)
    regime = regime_inicial
    regimes = np.zeros(n, dtype=int)
    regimes[0] = regime
    for i in range(1, n):
        L_M = a1 * t[i-1] + b1
        L_m = a2 * t[i-1] + b2
        if regime == 1 and x[i-1] >= L_M:
            regime = 2
        elif regime == 2 and x[i-1] <= L_m:
            regime = 1
        regimes[i] = regime
    return regimes


def _mle_por_regime(x, regimes, dt):
    '''Estimadores de máxima verosimilhança de (mu_r, sigma_r), condicionais a
    uma classificação em regimes fixa.

    O log-retorno log(x[k+1]/x[k]) é atribuído a regimes[k+1], o regime
    vigente no início desse intervalo. Os estimadores são
    anualizados pela divisão por dt, e a log-verosimilhança é a da densidade
    lognormal de X.

    Parâmetros
    ----------
    x : array
        Preços observados.
    regimes : array de inteiros em {1, 2}
        Regime em vigor em cada instante (mesmo comprimento de x).
    dt : float
        Passo temporal

    Devolve
    -------
    (resultados, log_lik_total):
        resultados : dict {1: {...}, 2: {...}}
            Por regime r, um dicionário com:
                'mu' : float ou None
                    Estimativa anualizada de mu_r; None se n_r < 2.
                'sigma' : float ou None
                    Estimativa anualizada de sigma_r; None se n_r < 2.
                'n_obs' : int
                    Número de log-retornos atribuídos ao regime r (n_r).
                'log_lik' : float
                    Contribuição do regime r para a log-verosimilhança
                    total; 0 se n_r < 2.
        log_lik_total : float
            Soma das contribuições 'log_lik' dos dois regimes.
    '''
    EPSILON = 1e-9
    log_retornos = np.diff(np.log(x))
    resultados = {}
    log_lik_total = 0.0
    for r in [1, 2]:
        no_regime = (regimes[1:] == r)
        z = log_retornos[no_regime]
        x_seguinte = x[1:][no_regime]
        n_r = len(z)
        if n_r > 1:
            sigma2 = max(np.var(z) / dt, EPSILON)        
            sigma = np.sqrt(sigma2)
            mu = np.mean(z) / dt + sigma2 / 2           
            residuos = z - (mu - 0.5 * sigma2) * dt
            log_lik = (- np.sum(np.log(x_seguinte))
                       + n_r * (-0.5 * np.log(2 * np.pi * sigma2 * dt))
                       - np.sum(residuos**2) / (2 * sigma2 * dt))
            log_lik_total += log_lik
            resultados[r] = {'mu': mu, 'sigma': sigma, 'n_obs': n_r, 'log_lik': log_lik}
        else:
            resultados[r] = {'mu': None, 'sigma': None, 'n_obs': n_r, 'log_lik': 0}
    return resultados, log_lik_total


def classificar_e_estimar(t, x, theta):
    '''Classifica a série em regimes com os limiares theta e estima (mu_r, sigma_r).

    Como o regime em t = 0 não é observável, testam-se ambos os regimes
    iniciais e retém-se a classificação com maior log-verosimilhança (a escolha
    só é relevante quando x[0] está entre os limiares).

    Parâmetros
    ----------
    t, x : array
        Instantes (anos) e preços observados.
    theta : sequência (a1, b1, a2, b2)
        Declives e ordenadas na origem de M(t) e m(t).

    Devolve
    -------
    (melhor_res, melhor_regimes, melhor_lik):
        melhor_res : dict {1: {...}, 2: {...}}
            As estimativas de mu, sigma, n_obs e log_lik de AMBOS os
            regimes, provenientes da classificação (regime_inicial=1 ou
            2) que obteve a maior log-verosimilhança total.
        melhor_regimes : array de inteiros em {1, 2}
            A classificação completa correspondente a essa mesma
            corrida vencedora, uma etiqueta por observação de x.
        melhor_lik : float
            A log-verosimilhança total dessa classificação (soma dos
            dois regimes).
    '''
    dt = t[1] - t[0]
    melhor_lik, melhor_res, melhor_regimes = -np.inf, None, None
    for regime_inicial in [1, 2]:
        regimes = _classificar_regimes(t, x, theta, regime_inicial)
        resultados, lik = _mle_por_regime(x, regimes, dt)
        if lik > melhor_lik:
            melhor_lik, melhor_res, melhor_regimes = lik, resultados, regimes
    return melhor_res, melhor_regimes, melhor_lik

## 3. Geração de sementes para a otimização

In [70]:
def _ajustar_retas(t_resist, x_resist, t_suporte, x_suporte, metodo, hiperparametros, T):
    '''Ajusta por mínimos quadrados a reta M(t) aos pontos (t_resist, x_resist)
    e a reta m(t) aos pontos (t_suporte, x_suporte).

    A semente só é aceite se M(t) > m(t) em todo o intervalo [0, T].

    Parâmetros
    ----------
    t_resist, x_resist : array
        Pontos de controlo de M(t).
    t_suporte, x_suporte : array
        Pontos de controlo de m(t).
    metodo : str
        Nome da estratégia de geração de sementes que produziu estes pontos
        (guardado na semente, apenas informativo).
    hiperparametros : dict
        Hiperparâmetros da estratégia (guardados na semente, apenas informativo).
    T : float
        Horizonte da série, em anos.

    Devolve
    -------
    dict ou None:
        Se a semente for aceite, um dicionário com:
            'metodo' : str
                O valor recebido em `metodo`.
            chaves de `hiperparametros`
                Os hiperparâmetros dessa estratégia, copiados tal como
                recebidos.
            'a1', 'b1' : float
                Declive e ordenada na origem da reta ajustada a
                (t_resist, x_resist) — M(t) = a1*t + b1.
            'a2', 'b2' : float
                Declive e ordenada na origem da reta ajustada a
                (t_suporte, x_suporte) — m(t) = a2*t + b2.
        None, se houver menos de dois pontos numa das retas, ou se a
        largura da banda ajustada não for positiva em todo o [0, T].
    '''
    if len(t_resist) < 2 or len(t_suporte) < 2:
        return None
    reta_resist = LinearRegression().fit(np.array(t_resist).reshape(-1, 1), x_resist)
    reta_suporte = LinearRegression().fit(np.array(t_suporte).reshape(-1, 1), x_suporte)
    theta = [reta_resist.coef_[0], reta_resist.intercept_,
             reta_suporte.coef_[0], reta_suporte.intercept_]
    if _largura_minima(theta, T) > 0:
        semente = {'metodo': metodo}
        semente.update(hiperparametros)
        semente.update({'a1': theta[0], 'b1': theta[1], 'a2': theta[2], 'b2': theta[3]})
        return semente
    return None


def _pontos_desfasados(t, x, indices, delta):
    '''Coordenadas (t, x) dos pontos em índices + delta, truncados ao domínio da série.

    Parâmetros
    ----------
    t, x : array
        Instantes e preços da série completa.
    indices : sequência de int
        Índices de referência (tipicamente extremos locais).
    delta : int
        Deslocamento temporal aplicado a cada índice, em observações.

    Devolve
    -------
    (t_desfasado, x_desfasado):
        t_desfasado : list de float
            t[i], onde i = índice de referência + delta, forçado a ficar
            dentro de [0, len(x)-1] — evita sair dos limites do array
            quando o índice deslocado cai fora da série.
        x_desfasado : list de float
            x[i], no mesmo índice i (já corrigido) de cada ponto.
    '''
    n = len(x)
    idx = [max(0, min(n - 1, p + delta)) for p in indices]
    return [t[i] for i in idx], [x[i] for i in idx]


def sementes_janela_fixa(t, x, w_list, alpha_list, delta_list):
    '''Estratégia A — extremos locais em janelas fixas.

    A série é dividida em janelas consecutivas de w observações. Em cada janela,
    são candidatos a M(t) os pontos com x >= (1 - alpha) * máximo e ao
    m(t) os pontos com x <= (1 + alpha) * mínimo. Os pontos são
    depois desfasados delta observações. Cada terno (w, alpha, delta)
    produz, no máximo, uma semente.

    Parâmetros
    ----------
    t, x : array
        Instantes e preços da série.
    w_list : sequência de int
        Comprimentos de janela a testar.
    alpha_list : sequência de float
        Margens de tolerância em torno do extremo de cada janela.
    delta_list : sequência de int
        Deslocamentos temporais a aplicar aos extremos.

    Devolve
    -------
    DataFrame, com uma linha por semente admissível e as colunas:
        'metodo' : str
            Sempre 'janela_fixa'.
        'w', 'alpha', 'delta' : int, float, int
            Os hiperparâmetros que geraram essa semente.
        'a1', 'b1', 'a2', 'b2' : float
            Declive e ordenada na origem de M(t) e de m(t) ajustadas.
        Pode ter menos linhas do que len(w_list) × len(alpha_list) ×
        len(delta_list): combinações sem pontos suficientes numa janela, ou
        com banda inadmissível.
    '''
    T = t[-1]
    sementes = []
    for w in w_list:
        for alpha in alpha_list:
            idx_max, idx_min = [], []
            for inicio in range(0, len(x), w):
                janela = x[inicio:inicio + w]
                if len(janela) < 2:
                    continue
                idx_max += [inicio + j for j in np.where(janela >= (1 - alpha) * np.max(janela))[0]]
                idx_min += [inicio + j for j in np.where(janela <= (1 + alpha) * np.min(janela))[0]]
            if not idx_max or not idx_min:
                continue
            for delta in delta_list:
                t_res, x_res = _pontos_desfasados(t, x, idx_max, delta)
                t_sup, x_sup = _pontos_desfasados(t, x, idx_min, delta)
                s = _ajustar_retas(t_res, x_res, t_sup, x_sup, 'janela_fixa',
                                   {'w': w, 'alpha': alpha, 'delta': delta}, T)
                if s:
                    sementes.append(s)
    return pd.DataFrame(sementes)


def sementes_janela_monotonica(t, x, w_list, alpha_list, delta_list):
    '''Estratégia B — como a estratégia A, mas os extremos de uma janela só são
    aproveitados se o seu máximo (mínimo) exceder (for inferior a) o da janela
    anterior, o que aproxima uma envolvente monotónica da série.

    Parâmetros
    ----------
    t, x : array
        Instantes e preços da série.
    w_list : sequência de int
        Comprimentos de janela a testar.
    alpha_list : sequência de float
        Margens de tolerância em torno do extremo de cada janela.
    delta_list : sequência de int
        Deslocamentos temporais a aplicar aos extremos.

    Devolve
    -------
    DataFrame, com uma linha por semente admissível e as colunas:
        'metodo' : str
            Sempre 'janela_monotonica'.
        'w', 'alpha', 'delta' : int, float, int
            Os hiperparâmetros que geraram essa semente.
        'a1', 'b1', 'a2', 'b2' : float
            Declive e ordenada na origem de M(t) e de m(t) ajustadas.
        Mesmo formato de sementes_janela_fixa, pode ter menos linhas do
        que len(w_list) × len(alpha_list) × len(delta_list).
    '''
    T = t[-1]
    sementes = []
    for w in w_list:
        for alpha in alpha_list:
            idx_max, idx_min = [], []
            max_anterior, min_anterior = -np.inf, np.inf
            for inicio in range(0, len(x), w):
                janela = x[inicio:inicio + w]
                if len(janela) < 2:
                    continue
                max_local, min_local = np.max(janela), np.min(janela)
                if max_local > max_anterior:
                    idx_max += [inicio + j for j in np.where(janela >= (1 - alpha) * max_local)[0]]
                if min_local < min_anterior:
                    idx_min += [inicio + j for j in np.where(janela <= (1 + alpha) * min_local)[0]]
                max_anterior, min_anterior = max_local, min_local
            if not idx_max or not idx_min:
                continue
            for delta in delta_list:
                t_res, x_res = _pontos_desfasados(t, x, idx_max, delta)
                t_sup, x_sup = _pontos_desfasados(t, x, idx_min, delta)
                s = _ajustar_retas(t_res, x_res, t_sup, x_sup, 'janela_monotonica',
                                   {'w': w, 'alpha': alpha, 'delta': delta}, T)
                if s:
                    sementes.append(s)
    return pd.DataFrame(sementes)


def sementes_percentil(t, x, w_list, p_superior=99, p_inferior=1):
    '''Estratégia C — percentis por janela.

    Em cada janela de w observações, são candidatos a M(t) os pontos
    acima do percentil p_superior e a m(t) os pontos abaixo do percentil
    p_inferior. Cada w produz, no máximo, uma semente.

    Parâmetros
    ----------
    t, x : array
        Instantes e preços da série.
    w_list : sequência de int
        Comprimentos de janela a testar.
    p_superior, p_inferior : float
        Percentis (0-100) que definem os pontos candidatos a M(t) e a
        m(t), respetivamente.

    Devolve
    -------
    DataFrame, com uma linha por semente admissível e as colunas:
        'metodo' : str
            Sempre 'percentil'.
        'w' : int
            O comprimento de janela que gerou essa semente.
        'p_superior', 'p_inferior' : float
            Os valores recebidos em `p_superior`/`p_inferior` — iguais
            em todas as linhas, já que não variam por semente.
        'a1', 'b1', 'a2', 'b2' : float
            Declive e ordenada na origem de M(t) e de m(t), ajustadas
            aos pontos de todas as janelas combinadas.
    '''
    T = t[-1]
    sementes = []
    for w in w_list:
        t_res, x_res, t_sup, x_sup = [], [], [], []
        for inicio in range(0, len(x), w):
            jx, jt = x[inicio:inicio + w], t[inicio:inicio + w]
            if len(jx) < 2:
                continue
            lim_sup = np.percentile(jx, p_superior)
            lim_inf = np.percentile(jx, p_inferior)
            for j in np.where(jx >= lim_sup)[0]:
                t_res.append(jt[j]); x_res.append(jx[j])
            for j in np.where(jx <= lim_inf)[0]:
                t_sup.append(jt[j]); x_sup.append(jx[j])
        s = _ajustar_retas(t_res, x_res, t_sup, x_sup, 'percentil',
                           {'w': w, 'p_superior': p_superior, 'p_inferior': p_inferior}, T)
        if s:
            sementes.append(s)
    return pd.DataFrame(sementes)


def sementes_media_movel(t, x, w_list, n_std=1.5):
    '''Estratégia D — bandas de Bollinger.

    Com média móvel MM e desvio-padrão móvel DP (janela centrada de w
    observações), são candidatos a M(t) os pontos acima de MM + n_std * DP
    e a m(t) os pontos abaixo de MM - n_std * DP.

    Parâmetros
    ----------
    t, x : array
        Instantes e preços da série.
    w_list : sequência de int
        Comprimentos da janela centrada da média e do desvio-padrão móveis.
    n_std : float
        Número de desvios-padrão móveis que definem a largura da banda.

    Devolve
    -------
    DataFrame, com uma linha por semente admissível e as colunas:
        'metodo' : str
            Sempre 'media_movel'.
        'w' : int
            O comprimento de janela que gerou essa semente.
        'n_std' : float
            O valor recebido em `n_std` — igual em todas as linhas.
        'a1', 'b1', 'a2', 'b2' : float
            Declive e ordenada na origem de M(t) e de m(t) ajustadas.
        Tem, no máximo, len(w_list) linhas (uma tentativa por w).
        '''
    T = t[-1]
    sementes = []
    serie = pd.Series(x)
    for w in w_list:
        media = serie.rolling(window=w, center=True).mean()
        desvio = serie.rolling(window=w, center=True).std()
        idx_max = np.where(x > (media + n_std * desvio).values)[0]
        idx_min = np.where(x < (media - n_std * desvio).values)[0]
        s = _ajustar_retas(t[idx_max], x[idx_max], t[idx_min], x[idx_min],
                           'media_movel', {'w': w, 'n_std': n_std}, T)
        if s:
            sementes.append(s)
    return pd.DataFrame(sementes)


def gerar_todas_sementes(t, x, w_list, alpha_list, delta_list,
                         p_sup_list, p_inf_list, n_std_list):
    '''Aplica as quatro estratégias com todas as combinações de hiperparâmetros e
    junta as sementes num único DataFrame.

    As estratégias A e B (janela fixa e monótona) recebem as listas de
    hiperparâmetros diretamente e tratam a grelha de combinações
    internamente. As estratégias C e D (percentil e
    média móvel) só aceitam um percentil/desvio de cada vez, pelo que esta
    função as chama repetidamente: uma vez por combinação de
    p_sup_list × p_inf_list para C, uma vez por valor de n_std_list para D.

    Parâmetros
    ----------
    t, x : array
        Instantes e preços da série.
    w_list, alpha_list, delta_list : sequências
        Hiperparâmetros das estratégias A e B.
    p_sup_list, p_inf_list : sequências
        Percentis superior e inferior da estratégia C.
    n_std_list : sequência
        Números de desvios-padrão da estratégia D.

    Devolve
    -------
    DataFrame com todas as sementes admissíveis das quatro estratégias,
    unidas por concatenação. Como cada estratégia produz colunas de
    hiperparâmetros diferentes ('alpha'/'delta' para A e B, 'p_superior'/
    'p_inferior' para C, 'n_std' para D), as colunas que não se aplicam a
    uma dada linha ficam a NaN. O 'metodo' de cada linha indica que
    colunas de hiperparâmetros lhe são relevantes. As colunas 'metodo',
    'a1', 'b1', 'a2', 'b2' estão sempre preenchidas. DataFrame vazio se
    nenhuma estratégia produzir sementes admissíveis.
    '''
    tabelas = [sementes_janela_fixa(t, x, w_list, alpha_list, delta_list),
               sementes_janela_monotonica(t, x, w_list, alpha_list, delta_list)]
    for p_sup in p_sup_list:
        for p_inf in p_inf_list:
            tabelas.append(sementes_percentil(t, x, w_list, p_sup, p_inf))
    for n_std in n_std_list:
        tabelas.append(sementes_media_movel(t, x, w_list, n_std))
    nao_vazias = [tab for tab in tabelas if not tab.empty]
    return pd.concat(nao_vazias, ignore_index=True) if nao_vazias else pd.DataFrame()

## 4. Estimação dos limiares afins

In [71]:
def definir_espaco_procura(df_sementes, margem=0.5):
    '''Define o espaço de procura a partir das sementes, alargado por `margem`.

    Os declives ficam em [-A, A], com A = (1 + margem) * max|a|
    As ordenadas na origem ficam entre o menor e o maior b das sementes, alargados em
    margem * |b|. Os mesmos limites aplicam-se às duas retas.

    Parâmetros
    ----------
    df_sementes : DataFrame
        Sementes geradas por gerar_todas_sementes (colunas a1, b1, a2, b2).
    margem : float
        Fração de folga acrescentada além do intervalo coberto pelas
        sementes (assume-se >= 0; um valor negativo encolhe o espaço
        abaixo do intervalo das sementes).

    Devolve
    -------
    (LIMITE_INF, LIMITE_SUP, AMPLITUDE):
        LIMITE_INF, LIMITE_SUP : array de 4 elementos
            Limites inferior e superior do espaço de procura, na ordem
            [a1, b1, a2, b2].
        AMPLITUDE : array de 4 elementos
            LIMITE_SUP - LIMITE_INF, elemento a elemento.
    '''
    global LIMITE_INF, LIMITE_SUP, AMPLITUDE
    a_max = max(abs(df_sementes[['a1', 'a2']].min().min()),
                abs(df_sementes[['a1', 'a2']].max().max())) * (1 + margem)
    b_min = df_sementes[['b1', 'b2']].min().min()
    b_max = df_sementes[['b1', 'b2']].max().max()
    b_inf = b_min - abs(b_min) * margem
    b_sup = b_max + abs(b_max) * margem
    LIMITE_INF = np.array([-a_max, b_inf, -a_max, b_inf])
    LIMITE_SUP = np.array([a_max, b_sup, a_max, b_sup])
    AMPLITUDE = LIMITE_SUP - LIMITE_INF
    dentro = np.ones(len(df_sementes), dtype=bool)
    for j, p in enumerate(['a1', 'b1', 'a2', 'b2']):
        dentro &= df_sementes[p].between(LIMITE_INF[j], LIMITE_SUP[j]).values
    print(f"Sementes dentro do espaço de procura: {dentro.sum()}/{len(df_sementes)}")
    return LIMITE_INF, LIMITE_SUP, AMPLITUDE


def normalizar(theta):
    '''Converte theta da escala original para [0, 1]^4.

    Parâmetros
    ----------
    theta : array de 4 elementos
        Vetor (a1, b1, a2, b2) na escala original.

    Devolve
    -------
    Array de 4 elementos em [0, 1], relativo aos limites definidos por
    definir_espaco_procura.
    '''
    return (theta - LIMITE_INF) / AMPLITUDE


def desnormalizar(theta_norm):
    '''Converte theta_norm de [0, 1]^4 para a escala original.

    Parâmetros
    ----------
    theta_norm : array de 4 elementos
        Vetor (a1, b1, a2, b2) normalizado em [0, 1].

    Devolve
    -------
    Array de 4 elementos (a1, b1, a2, b2) na escala original.
    '''
    return theta_norm * AMPLITUDE + LIMITE_INF


def _largura_minima(theta, T):
    '''Menor valor de M(t) - m(t) em [0, T].

    Sendo M - m afim, o mínimo é atingido num dos extremos do intervalo.

    Parâmetros
    ----------
    theta : sequência (a1, b1, a2, b2)
        Declives e ordenadas na origem de M(t) e m(t).
    T : float
        Fim do intervalo considerado, em anos.

    Devolve
    -------
    float : largura mínima da banda em [0, T].
    '''
    a1, b1, a2, b2 = theta
    return min(b1 - b2, (a1 * T + b1) - (a2 * T + b2))


def _suporte_minimo(theta, T):
    '''Menor valor de m(t) em [0, T] (atingido num dos extremos, por m ser afim).

    Parâmetros
    ----------
    theta : sequência (a1, b1, a2, b2)
        Declives e ordenadas na origem de M(t) e m(t).
    T : float
        Fim do intervalo considerado, em anos.

    Devolve
    -------
    float : valor mínimo de m(t) em [0, T].
    '''
    _, _, a2, b2 = theta
    return min(b2, a2 * T + b2)


def simplex_regular(n, gamma, centro):
    '''Simplex regular de n + 1 vértices em R^n, com escala gamma, deslocado para.

    Parâmetros
    ----------
    n : int
        Dimensão do espaço (número de parâmetros).
    gamma : float
        Escala do simplex.
    centro : array de n elementos
        Ponto para o qual o simplex é deslocado.

    Devolve
    -------
    array (n + 1, n)
        Uma linha por vértice do simplex (n + 1 vértices), uma coluna por
        coordenada em R^n. Cada linha é um ponto candidato inicial,
        já escalado por `gamma` e deslocado para `centro` — a matriz
    '''
    Y = np.zeros((n + 1, n))
    soma_quadrados = 0
    for i in range(n):
        c = np.sqrt(1 - soma_quadrados)
        r = (-1/n - soma_quadrados) / c
        Y[i, i] = c
        for j in range(i + 1, n + 1):
            Y[j, i] = r
        soma_quadrados += r**2
    for i in range(n + 1):
        Y[i] = gamma * Y[i] + centro
    return Y


def _objetivo_normalizado(theta_norm, t, x, largura_min=0.0):
    '''Função a minimizar: -log L(theta), com theta_norm em escala normalizada.

    Configurações inadmissíveis recebem uma penalização da ordem de 1e12,
    crescente com a violação (para orientar o simplex de volta à região
    admissível):
      - theta_norm fora de [0, 1]^4;
      - m(t) não positivo algures em [0, T];
      - largura da banda M(t) - m(t) inferior a largura_min algures em [0, T].

    Parâmetros
    ----------
    theta_norm : array de 4 elementos
        Candidato (a1, b1, a2, b2) normalizado em [0, 1]^4.
    t, x : array
        Instantes e preços da série.
    largura_min : float
        Largura mínima admissível da banda M(t) - m(t).

    Devolve
    -------
    float
        -log-verosimilhança do candidato, se admissível.
        Se inadmissível, uma penalização >= 1e12, tanto maior quanto mais
        grave a violação — sempre muito superior a qualquer
        -log-verosimilhança real, para nunca ser confundida com uma
        solução genuína pelo otimizador.
    '''
    PENALIZACAO = 1e12
    excesso = (np.sum(np.clip(-theta_norm, 0, None))
               + np.sum(np.clip(theta_norm - 1, 0, None)))
    if excesso > 0:
        return PENALIZACAO * (1.0 + excesso)
    theta = desnormalizar(theta_norm)
    T = t[-1]
    suporte_min = _suporte_minimo(theta, T)
    if suporte_min <= 0:
        return PENALIZACAO * (1.0 + abs(suporte_min))
    largura = _largura_minima(theta, T)
    if largura < largura_min:
        return PENALIZACAO * (1.0 + abs(largura - largura_min))
    _, _, log_lik = classificar_e_estimar(t, x, list(theta))
    return -log_lik


def estimar_limiares_afins(t, x, df_sementes, gamma=0.1, xatol=1e-4, fatol=1e-4,
                               maxiter=500, kappa_min=3):
    '''Nelder-Mead multi-start sobre as sementes.

    Cada semente dentro do espaço de procura é ponto de partida de um
    Nelder-Mead com simplex inicial regular de escala gamma (em [0, 1]^4). A
    largura mínima da banda é kappa_min movimentos diários típicos do preço
    (ver _variacao_diaria_tipica).

    Parâmetros
    ----------
    t, x : array
        Instantes e preços da série de treino.
    df_sementes : DataFrame
        Sementes geradas por gerar_todas_sementes.
    gamma : float
        Escala do simplex inicial do Nelder-Mead.
    xatol, fatol : float
        Tolerâncias de paragem do Nelder-Mead (dispersão dos vértices e da
        função objetivo, respetivamente).
    maxiter : int
        Número máximo de iterações por execução do Nelder-Mead.
    kappa_min : float
        Largura mínima da banda, em movimentos diários típicos do preço.

    Devolve
    -------
    (theta_final, log_lik, semente_vencedora):
        theta_final : array de 4 elementos, ou None
            O (a1, b1, a2, b2) de maior log-verosimilhança entre todas as
            sementes testadas, na escala original.
        log_lik : float ou None
            A log-verosimilhança de theta_final.
        semente_vencedora : dict, ou None
            A semente (linha de df_sementes) que deu origem a
            theta_final, com a chave adicional 'LogLik_final'.
    '''
    melhor_objetivo = np.inf
    theta_final, semente_vencedora = None, None
    largura_min = kappa_min * _variacao_diaria_tipica(x)
    for _, semente in df_sementes.iterrows():
        inicio = normalizar(np.array([semente['a1'], semente['b1'],
                                      semente['a2'], semente['b2']]))
        if np.any(inicio < 0) or np.any(inicio > 1):
            continue
        res = minimize(_objetivo_normalizado, inicio, args=(t, x, largura_min),
                       method='Nelder-Mead',
                       options={'initial_simplex': simplex_regular(4, gamma, inicio),
                                'xatol': xatol, 'fatol': fatol, 'maxiter': maxiter})
        if res.fun < melhor_objetivo:
            melhor_objetivo = res.fun
            theta_final = desnormalizar(res.x)
            semente_vencedora = semente.to_dict()
            semente_vencedora['LogLik_final'] = round(-res.fun, 4)
    log_lik = -melhor_objetivo if theta_final is not None else None
    return theta_final, log_lik, semente_vencedora

## 5. Modelos de referência: MBG e limiares constantes

In [72]:
def _estimar_mbg(t, x):
    '''Estimação por máxima verosimilhança do MBG com um único regime.

    A log-verosimilhança é a da densidade lognormal de X.

    Parâmetros
    ----------
    t, x : array
        Instantes e preços da série.

    Devolve
    -------
    (log_lik, mu, sigma2):
        log_lik : float
            Log-verosimilhança maximizada.
        mu : float
            Estimativa anualizada da tendência.
        sigma2 : float
            Estimativa anualizada da variância.
    '''
    dt = t[1] - t[0]
    log_retornos = np.diff(np.log(x))
    n = len(log_retornos)
    sigma2 = np.var(log_retornos) / dt
    mu = np.mean(log_retornos) / dt + sigma2 / 2
    residuos = log_retornos - (mu - 0.5 * sigma2) * dt
    log_lik = (- np.sum(np.log(x[1:]))
               + n * (-0.5 * np.log(2 * np.pi * sigma2 * dt))
               - np.sum(residuos**2) / (2 * sigma2 * dt))
    return log_lik, mu, sigma2


def _classificar_regimes_fixos(x, M, m, regime_inicial=1):
    '''Classificação em regimes com limiares horizontais M e m. Equivale a
    _classificar_regimes com a1 = a2 = 0.

    Parâmetros
    ----------
    x : array
        Preços observados.
    M, m : float
        Limiares horizontais superior e inferior.
    regime_inicial : int
        Regime assumido no primeiro instante.

    Devolve
    -------
    regimes : array de inteiros em {1, 2}
        Um por observação de x, na mesma convenção de _classificar_regimes:
        regimes[k] é o regime vigente no início do intervalo que gerou o
        retorno terminado em x[k].
    '''
    n = len(x)
    regime = regime_inicial
    regimes = np.zeros(n, dtype=int)
    regimes[0] = regime
    for i in range(1, n):
        if regime == 1 and x[i-1] >= M:
            regime = 2
        elif regime == 2 and x[i-1] <= m:
            regime = 1
        regimes[i] = regime
    return regimes


def _variacao_diaria_tipica(x):
    '''Variação diária típica do preço, em unidades monetárias.

    Parâmetros
    ----------
    x : array
        Preços observados.

    Devolve
    -------
    float
        O desvio-padrão dos log-retornos vezes a mediana do preço
        Uma aproximação do movimento diário típico, em unidades monetárias.
        0.0 se x tiver 2 ou menos observações.
    '''
    log_retornos = np.diff(np.log(x))
    if len(log_retornos) < 2:
        return 0.0
    return float(np.std(log_retornos)) * float(np.median(x))


def _estimar_limiares_fixos(t, x, n_grid=100, min_frac_regime=0.0, kappa_min=3):
    '''Estimação do modelo de limiares constantes por pesquisa em grelha sobre (M, m).

    Implementação prática dos estimadores, em duas
    fases: (i) grelha com n_grid quantis da série entre os percentis 5 e 95;
    (ii) refinamento com n_grid x n_grid pontos em torno do melhor par.

    Um par (M, m) é admissível se:
      - m < M e M - m >= kappa_min * variação diária típica, o que exclui
        comutações que traduzem apenas ruído de curto prazo;
      - cada regime tem pelo menos duas observações e, pelo menos, uma fração
        min_frac_regime da amostra.

    Parâmetros
    ----------
    t, x : array
        Instantes e preços da série de treino.
    n_grid : int
        Número de pontos da grelha, em cada uma das duas fases.
    min_frac_regime : float
        Fração mínima da amostra exigida em cada regime.
    kappa_min : float
        Largura mínima da banda, em movimentos diários típicos do preço.

    Devolve
    -------
    (theta_fixo, log_lik, M, m):
        theta_fixo : list [0.0, M, 0.0, m], ou None
            Os limiares constantes na mesma representação de theta
            usada no resto do código (a1 = a2 = 0, b1 = M, b2 = m).
            None se nenhum par (M, m) for admissível.
        log_lik : float
            A log-verosimilhança do par (M, m) devolvido. -inf se
            nenhum par for admissível.
        M, m : float, ou None
            Os limiares superior e inferior encontrados — os mesmos
            valores já embutidos em theta_fixo. None se nenhum par
            for admissível.
    '''
    dt = t[1] - t[0]
    n = len(x)
    largura_min = kappa_min * _variacao_diaria_tipica(x)

    def _log_lik_par(M, m):
        '''Log-verosimilhança do par de limiares constantes (M, m). Uso interno de
        _estimar_limiares_fixos: testa os dois regimes iniciais possíveis e
        devolve o maior valor entre classificações admissíveis, ou -inf.

        Parâmetros
        ----------
        M, m : float
            Limiares superior e inferior candidatos.

        Devolve
        -------
        float (melhor) : log-verosimilhança do melhor regime inicial, ou -inf
        se o par for inadmissível (m >= M, banda demasiado estreita, ou
        nenhum regime inicial produzir dois regimes com observações
        suficientes).
        '''
        if m >= M or (M - m) < largura_min:
            return -np.inf
        melhor = -np.inf
        for regime_inicial in [1, 2]:
            regimes = _classificar_regimes_fixos(x, M, m, regime_inicial)
            n1 = int(np.sum(regimes[1:] == 1))
            n2 = int(np.sum(regimes[1:] == 2))
            if n1 < 2 or n2 < 2:                           
                continue
            if n1 < min_frac_regime * n or n2 < min_frac_regime * n:
                continue
            _, lik = _mle_por_regime(x, regimes, dt)
            melhor = max(melhor, lik)
        return melhor

    # Fase 1: grelha de quantis
    x_ordenado = np.sort(x)
    grelha = np.unique(x_ordenado[np.linspace(int(0.05 * n), int(0.95 * n), n_grid, dtype=int)])
    if len(grelha) < 2:
        return None, -np.inf, None, None

    melhor_lik, M_hat, m_hat = -np.inf, None, None
    for i, M in enumerate(grelha):
        for m in grelha[:i]:                                 
            lik = _log_lik_par(M, m)
            if lik > melhor_lik:
                melhor_lik, M_hat, m_hat = lik, M, m
    if M_hat is None:
        return None, -np.inf, None, None

    # Fase 2: refinamento entre os vizinhos do ótimo na grelha
    def _grelha_fina(valor):
        '''Intervalo de refinamento em torno de `valor`, robusto a grelhas curtas.
        Uso interno de _estimar_limiares_fixos, na segunda fase da pesquisa.

        Parâmetros
        ----------
        valor : float
            Ponto em torno do qual se constrói a grelha refinada (tipicamente o
            melhor M ou m da primeira fase).

        Devolve
        -------
        array : grelha densa de n_grid pontos em torno de `valor`.
        '''
        i = int(np.searchsorted(grelha, valor))
        inf = grelha[max(0, i - 1)]
        sup = grelha[min(len(grelha) - 1, i + 1)]
        if sup <= inf:                                     
            passo = float(np.ptp(grelha)) / max(1, len(grelha) - 1)
            inf, sup = valor - passo, valor + passo
        return np.linspace(inf, sup, n_grid)

    for M in _grelha_fina(M_hat):
        for m in _grelha_fina(m_hat):
            if m >= M:
                continue
            lik = _log_lik_par(M, m)
            if lik > melhor_lik:
                melhor_lik, M_hat, m_hat = lik, M, m
    return [0.0, float(M_hat), 0.0, float(m_hat)], melhor_lik, M_hat, m_hat

## 6. Comparação de modelos (AIC/BIC)

In [73]:
def comparar_modelos(t, x, theta_afim, lik_afim, kappa_min=3):
    '''Compara por AIC e BIC o MBG (k = 2), os limiares constantes (k = 6) e os
    limiares afins (k = 8).

    O MBG e os limiares constantes são estimados aqui. O modelo afim não é
    reestimado: usam-se theta_afim e lik_afim já obtidos (tipicamente após
    selecionar_melhor_candidato), para que seja avaliado na sua máxima
    verosimilhança efetiva. Se não houver nenhum par (M, m) admissível, os
    limiares constantes são omitidos da tabela.

    Parâmetros
    ----------
    t, x : array
        Instantes e preços da série de treino.
    theta_afim : sequência (a1, b1, a2, b2)
        Limiares afins já estimados.
    lik_afim : float
        Log-verosimilhança de theta_afim.
    kappa_min : float
        Largura mínima da banda, em movimentos diários típicos do preço
        (aplicada também à estimação dos limiares constantes).

    Devolve
    -------
    (tabela, theta_fixo):
        tabela : DataFrame
            Uma linha por modelo (MBG; Limiares Constantes, se admissível;
            Limiares Afins), com colunas k, Log-Lik, AIC, BIC, indicadores
            booleanos do melhor AIC/BIC, e Δ AIC / Δ BIC face ao melhor
            valor de cada critério.
        theta_fixo : list [0.0, M_hat, 0.0, m_hat], ou None
            Os limiares constantes estimados por _estimar_limiares_fixos.
            None exatamente quando nenhum par (M, m) foi admissível.
    '''
    n_obs = len(x) - 1

    lik_mbg, mu_mbg, sigma2_mbg = _estimar_mbg(t, x)
    print("  MBG:")
    print(f"    mu={mu_mbg:.6f}, sigma={np.sqrt(sigma2_mbg):.6f}, LogLik={lik_mbg:.4f}")

    print("  Limiares constantes (pesquisa em grelha):")
    theta_fixo, lik_fixo, M_hat, m_hat = _estimar_limiares_fixos(t, x, kappa_min=kappa_min)
    if M_hat is not None:
        print(f"    M={M_hat:.4f}, m={m_hat:.4f}, LogLik={lik_fixo:.4f}")
    else:
        print("    nenhum par (M, m) admissível: modelo omitido")

    a1, b1, a2, b2 = theta_afim
    print("  Limiares afins (estimativa fornecida):")
    print(f"    a1={a1:.4f}, b1={b1:.4f}, a2={a2:.4f}, b2={b2:.4f}, LogLik={lik_afim:.4f}")

    def aic(k, ll): 
        '''Critério de informação de Akaike. Uso interno de comparar_modelos.

        Parâmetros
        ----------
        k : int
            Número de parâmetros do modelo.
        ll : float
            Log-verosimilhança maximizada.

        Devolve
        -------
        float : AIC = 2k - 2*ll.
        '''
        return 2 * k - 2 * ll
    def bic(k, ll): 
        '''Critério de informação bayesiano. Uso interno de comparar_modelos.

        Parâmetros
        ----------
        k : int
            Número de parâmetros do modelo.
        ll : float
            Log-verosimilhança maximizada.

        Devolve
        -------
        float : BIC = k*log(n_obs) - 2*ll (n_obs capturado do âmbito envolvente).
        '''
        return k * np.log(n_obs) - 2 * ll

    modelos = [('MBG', 2, lik_mbg), ('Limiares Afins', 8, lik_afim)]
    if M_hat is not None and np.isfinite(lik_fixo):
        modelos.insert(1, ('Limiares Constantes', 6, lik_fixo))

    tabela = pd.DataFrame([{'Modelo': nome, 'k': k, 'Log-Lik': round(float(ll), 4),
                            'AIC': round(float(aic(k, ll)), 2),
                            'BIC': round(float(bic(k, ll)), 2)}
                           for nome, k, ll in modelos])
    tabela['Melhor AIC'] = tabela['AIC'] == tabela['AIC'].min()
    tabela['Melhor BIC'] = tabela['BIC'] == tabela['BIC'].min()
    tabela['Δ AIC'] = (tabela['AIC'] - tabela['AIC'].min()).round(2)
    tabela['Δ BIC'] = (tabela['BIC'] - tabela['BIC'].min()).round(2)
    return tabela, theta_fixo

## 7. Diagnóstico dos parâmetros por regime

In [74]:
def diagnosticar_regimes(t, x, theta, nivel=0.05, verbose=True):
    '''Estimativas, erros-padrão e intervalos de confiança de (mu_r, sigma_r),
    e testes à diferença entre regimes.

    Erros-padrão assintóticos, condicionais à classificação em regimes:
      SE(mu)    = sigma / sqrt(T_r);
      SE(sigma) = sigma / sqrt(2 n_r).
    O IC de mu usa a aproximação normal.
    O IC de sigma usa a distribuição qui-quadrado com n_r - 1 graus de liberdade.

    Testes entre regimes:
      - diferença de drift: teste z;
      - diferença de volatilidade: rácio sigma1/sigma2 e teste de Levene;
      - sobreposição dos IC de mu e de sigma;
      - rótulo qualitativo, atribuído pela primeira condição satisfeita:
          1. 'regime residual' — um regime tem menos de 5% das
             observações válidas ou menos de 20 observações;
          2. 'segregação de volatilidade' — sigma1/sigma2 > 2 ou < 0.5,
             e o teste de Levene rejeita igualdade de variâncias
             (p < 0.05);
          3. 'alternância de tendência' — o teste z rejeita igualdade
             de drift (p <= 0.05);
          4. 'regime indistinto' — nenhuma das condições anteriores.

    Parâmetros
    ----------
    t, x : array
        Instantes e preços da série.
    theta : sequência (a1, b1, a2, b2)
        Limiares usados para classificar os regimes.
    nivel : float
        Nível de significância dos testes e dos intervalos de confiança
        (1 - nível é o grau de confiança).
    verbose : bool
        Se True, imprime as tabelas e a classificação qualitativa.

    Devolve
    -------
    (tabela, testes):
        tabela : DataFrame
            Uma linha por regime (1 e 2). Para um regime com n_r
            observações válidas: N_obs, Tempo (anos), mu, IC de mu,
            SE_mu, sigma, IC de sigma, SE_sigma.
        testes : dict
            As comparações entre regimes (mu1 - mu2, z_mu, p_mu,
            sigma1/sigma2, p_Levene, sobreposição dos IC de mu e de
            sigma, e o rótulo qualitativo).
    '''
    resultados, regimes, log_lik = classificar_e_estimar(t, x, theta)
    dt = t[1] - t[0]
    log_retornos = np.diff(np.log(x))
    z_crit = stats.norm.ppf(1 - nivel / 2)

    linhas, retornos_regime = {}, {}
    for r in [1, 2]:
        mu, sigma, n_r = resultados[r]['mu'], resultados[r]['sigma'], resultados[r]['n_obs']
        if mu is None or sigma is None or n_r < 3:
            linhas[r] = {'Regime': r, 'N_obs': n_r, 'Aviso': 'observações insuficientes'}
            continue
        retornos_regime[r] = log_retornos[regimes[1:] == r]
        T_r = n_r * dt
        se_mu = sigma / np.sqrt(T_r)
        se_sigma = sigma / np.sqrt(2 * n_r)
        gl = n_r - 1
        linhas[r] = {
            'Regime': r, 'N_obs': n_r, 'Tempo (anos)': round(T_r, 2),
            'mu (anual)': round(mu, 4),
            'IC_mu_inf': round(mu - z_crit * se_mu, 4),
            'IC_mu_sup': round(mu + z_crit * se_mu, 4),
            'SE_mu': round(se_mu, 4),
            'sigma (anual)': round(sigma, 4),
            'IC_sigma_inf': round(sigma * np.sqrt(n_r / chi2.ppf(1 - nivel / 2, gl)), 4),
            'IC_sigma_sup': round(sigma * np.sqrt(n_r / chi2.ppf(nivel / 2, gl)), 4),
            'SE_sigma': round(se_sigma, 4),
        }
    tabela = pd.DataFrame([linhas[r] for r in [1, 2]])

    testes = {}
    if 1 in retornos_regime and 2 in retornos_regime:
        mu1, mu2 = resultados[1]['mu'], resultados[2]['mu']
        s1, s2 = resultados[1]['sigma'], resultados[2]['sigma']
        n1, n2 = resultados[1]['n_obs'], resultados[2]['n_obs']
        se_mu1, se_mu2 = s1 / np.sqrt(n1 * dt), s2 / np.sqrt(n2 * dt)
        se_s1, se_s2 = s1 / np.sqrt(2 * n1), s2 / np.sqrt(2 * n2)

        z_mu = (mu1 - mu2) / np.sqrt(se_mu1**2 + se_mu2**2)
        p_mu = 2 * (1 - stats.norm.cdf(abs(z_mu)))
        p_levene = stats.levene(retornos_regime[1], retornos_regime[2]).pvalue
        racio_sigma = s1 / s2

        def _sobrepoem(c1, se1, c2, se2):
            '''Verifica se dois intervalos de confiança normais se sobrepõem. Uso interno
            de diagnosticar_regimes.

            Parâmetros
            ----------
            c1, c2 : float
                Estimativas centrais dos dois regimes.
            se1, se2 : float
                Erros-padrão associados a c1 e c2.

            Devolve
            -------
            bool : True se os intervalos [c - z*se, c + z*se] dos dois regimes se
            sobrepuserem (z fixado no âmbito envolvente).
            '''
            inf1, sup1 = c1 - z_crit * se1, c1 + z_crit * se1
            inf2, sup2 = c2 - z_crit * se2, c2 + z_crit * se2
            return not (sup1 < inf2 or sup2 < inf1)

        sobrepoe_mu = _sobrepoem(mu1, se_mu1, mu2, se_mu2)
        sobrepoe_sigma = _sobrepoem(s1, se_s1, s2, se_s2)

        fracao_menor = min(n1, n2) / (n1 + n2)
        if fracao_menor < 0.05 or min(n1, n2) < 20:
            rotulo = 'regime residual'
        elif (racio_sigma > 2 or racio_sigma < 0.5) and p_levene < 0.05:
            rotulo = 'segregação de volatilidade'
        elif p_mu <= 0.05:
            rotulo = 'alternância de tendência'
        else:
            rotulo = 'regime indistinto'

        testes = {'mu1 - mu2': round(mu1 - mu2, 4), 'z_mu': round(z_mu, 3),
                  'p_mu': round(p_mu, 4), 'sigma1 / sigma2': round(racio_sigma, 3),
                  'p_Levene': round(p_levene, 4),
                  'IC_mu_sobrepoe': bool(sobrepoe_mu),
                  'IC_sigma_sobrepoe': bool(sobrepoe_sigma),
                  'rotulo': rotulo}

    if verbose:
        print(f"Log-verosimilhança da classificação: {log_lik:.2f}\n")
        print("--- Estimativas por regime ---")
        print(tabela.to_string(index=False))
        if testes:
            print("\n--- Diferença entre regimes ---")
            print(f"  Drift:        mu1-mu2 = {testes['mu1 - mu2']:+.4f} "
                  f"(z={testes['z_mu']}, p={testes['p_mu']})  IC sobrepõem-se: {testes['IC_mu_sobrepoe']}")
            print(f"  Volatilidade: s1/s2   = {testes['sigma1 / sigma2']} "
                  f"(Levene p={testes['p_Levene']})  IC sobrepõem-se: {testes['IC_sigma_sobrepoe']}")
            print(f"  Classificação: {testes['rotulo'].upper()}")
        print("\n  NOTA: valores-p condicionais à classificação estimada (otimistas). A evidência")
        print("  confirmatória de que os regimes acrescentam algo é a comparação AIC/BIC com o MBG.")
    return tabela, testes

## 8. Identificabilidade

In [75]:
_INDICE_PARAMETRO = {'a1': 0, 'b1': 1, 'a2': 2, 'b2': 3}


def mapa_identificabilidade(t, x, theta_otimo, parametros=('a1', 'b1'), n_grid=40,
                            semiamplitudes=None, delta_ll=2.0, c_amp=0.25, largura_min=0.0,
                            mostrar=True, guardar=None, titulo=None):
    '''Perfil bidimensional da log-verosimilhança em torno do ótimo.

    Os dois parâmetros indicados em `parametros` variam numa grelha
    n_grid x n_grid centrada no ótimo, enquanto os restantes ficam fixos. As
    semiamplitudes por omissão são proporcionais à amplitude A da série:
    c_amp * A para as ordenadas na origem e c_amp * A / T para os declives, de
    modo que ambas desloquem a fronteira da mesma quantidade em t = T. (Se a
    série for quase constante, A é substituída por um valor mínimo de
    salvaguarda.) Podem ser impostas por
    parâmetro através de `semiamplitudes`.

    Pontos inadmissíveis (m(t) não positivo ou banda mais estreita do que
    largura_min) ficam a NaN, tal como qualquer ponto em que a classificação
    e estimação em `classificar_e_estimar` falhe.
    O gráfico assinala o ótimo e o contorno
    LL = max(LL) - delta_ll.


    Parâmetros
    ----------
    t, x : array
        Instantes e preços da série.
    theta_otimo : sequência (a1, b1, a2, b2)
        Solução de referência, em torno da qual se constrói o mapa.
    parametros : tuple de 2 str
        Nomes dos dois parâmetros a variar, de entre 'a1', 'b1', 'a2', 'b2'.
    n_grid : int
        Número de pontos da grelha, em cada eixo.
    semiamplitudes : dict, opcional
        Semiamplitude a impor a cada parâmetro.
    delta_ll : float
        Definição do contorno de tolerância desenhado no gráfico.
    c_amp : float
        Fração da amplitude da série usada nas semiamplitudes.
    largura_min : float
        Largura mínima admissível da banda M(t) - m(t).
    mostrar : bool
        Se True, mostra o gráfico.
    guardar : str, opcional
        Caminho do ficheiro onde o gráfico é guardado.
    titulo : str, opcional
        Título do gráfico.

    Devolve
    -------
    (grelha1, grelha2, grelha_ll): matrizes n_grid x n_grid,
    com os valores testados do primeiro parâmetro (grelha1),
    do segundo (grelha2) e a log-verosimilhança correspondente
    (grelha_ll).
    '''
    theta_otimo = list(map(float, theta_otimo))
    T = float(t[-1])
    p1, p2 = parametros
    i1, i2 = _INDICE_PARAMETRO[p1], _INDICE_PARAMETRO[p2]
    v1, v2 = theta_otimo[i1], theta_otimo[i2]
    amplitude_serie = float(np.max(x) - np.min(x))
    if not np.isfinite(amplitude_serie) or amplitude_serie <= 0:
        amplitude_serie = max(1.0, abs(float(np.mean(x))) * 0.1)

    def _semiamplitude(nome):
        '''Semiamplitude da grelha para um parâmetro. Uso interno de
        mapa_identificabilidade: devolve o valor imposto em `semiamplitudes`, se
        indicado, ou o valor por omissão (c_amp * amplitude, ajustado por T para
        os declives).

        Parâmetros
        ----------
        nome : str
            Nome do parâmetro ('a1', 'b1', 'a2' ou 'b2').

        Devolve
        -------
        float : semiamplitude do parâmetro, na escala original.
        '''
        if semiamplitudes and nome in semiamplitudes:
            return float(semiamplitudes[nome])
        if nome in ('a1', 'a2'):
            return c_amp * amplitude_serie / max(T, 1e-9)
        return c_amp * amplitude_serie

    eixo1 = np.linspace(v1 - _semiamplitude(p1), v1 + _semiamplitude(p1), n_grid)
    eixo2 = np.linspace(v2 - _semiamplitude(p2), v2 + _semiamplitude(p2), n_grid)
    grelha1, grelha2 = np.meshgrid(eixo1, eixo2)
    grelha_ll = np.full(grelha1.shape, np.nan)
    for i in range(n_grid):
        for j in range(n_grid):
            theta = theta_otimo.copy()
            theta[i1], theta[i2] = grelha1[i, j], grelha2[i, j]
            if _suporte_minimo(theta, T) <= 0 or _largura_minima(theta, T) < largura_min:
                continue
            try:
                grelha_ll[i, j] = classificar_e_estimar(t, x, theta)[2]
            except Exception:
                pass

    if mostrar or guardar:
        ll_max = np.nanmax(grelha_ll)
        fig, ax = plt.subplots(figsize=(8, 6.5))
        cmap = plt.cm.viridis.copy()
        cmap.set_bad('lightgrey')
        malha = ax.pcolormesh(eixo1, eixo2, np.ma.masked_invalid(grelha_ll),
                              cmap=cmap, shading='auto')
        try:
            ax.contour(eixo1, eixo2, np.ma.masked_invalid(grelha_ll),
                       levels=[ll_max - delta_ll], colors='white', linewidths=1.2)
        except Exception:
            pass
        ax.plot(v1, v2, marker='*', color='red', markersize=16,
                markeredgecolor='white', label='ótimo')
        fig.colorbar(malha, ax=ax, label='log-verosimilhança')
        ax.set_xlabel(p1); ax.set_ylabel(p2)
        ax.set_title(titulo or f'Identificabilidade: {p1} vs {p2}')
        ax.legend(loc='upper right')
        plt.tight_layout()
        if guardar:
            plt.savefig(guardar, dpi=110, bbox_inches='tight')
        if mostrar:
            plt.show()
        plt.close(fig)
    return grelha1, grelha2, grelha_ll


def _refinar_localmente(t, x, theta_inicial, gamma=0.1, largura_min=0.0):
    '''Nelder-Mead local a partir de theta_inicial, com tolerâncias mais finas
    do que na estimação principal.

    Parâmetros
    ----------
    t, x : array
        Instantes e preços da série.
    theta_inicial : sequência (a1, b1, a2, b2)
        Ponto de partida do refinamento.
    gamma : float
        Escala do simplex inicial.
    largura_min : float
        Largura mínima admissível da banda M(t) - m(t).

    Devolve
    -------
    (theta, log_lik):
        theta : array (a1, b1, a2, b2)
            O ótimo local encontrado pelo Nelder-Mead, já desnormalizado
            para a escala original.
        log_lik : float
            A log-verosimilhança nesse ótimo.
        Ou (None, -inf), se theta_inicial estiver fora do espaço de
        procura, isto é, fora do hipercubo normalizado [0, 1]^4.
    '''
    inicio = normalizar(np.array(theta_inicial, float))
    if np.any(inicio < 0) or np.any(inicio > 1):
        return None, -np.inf
    T = t[-1]

    def _objetivo(theta_norm):
        '''Log-verosimilhança negada de theta_norm, para o Nelder-Mead local. Uso
        interno de _refinar_localmente: devolve um valor muito elevado (1e18) se
        theta_norm estiver fora do hipercubo normalizado ou a configuração for
        inadmissível.

        Parâmetros
        ----------
        theta_norm : array de 4 elementos
            Candidato (a1, b1, a2, b2) normalizado em [0, 1]^4.

        Devolve
        -------
        float : -log-verosimilhança (ou penalização, se inadmissível).
        '''
        if np.any(theta_norm < 0) or np.any(theta_norm > 1):
            return 1e18
        theta = desnormalizar(theta_norm)
        if _suporte_minimo(theta, T) <= 0 or _largura_minima(theta, T) < largura_min:
            return 1e18
        return -classificar_e_estimar(t, x, list(theta))[2]

    res = minimize(_objetivo, inicio, method='Nelder-Mead',
                   options={'initial_simplex': simplex_regular(4, gamma, inicio),
                            'xatol': 1e-5, 'fatol': 1e-5, 'maxiter': 800})
    return desnormalizar(res.x), -res.fun


def selecionar_melhor_candidato(t, x, candidatos, refinar=True, gamma=0.1,
                                largura_min=0.0, verbose=True):
    '''Devolve o candidato admissível de maior log-verosimilhança.

    Cada candidato é avaliado tal como está e, se refinar=True, também após
    refinamento local. Candidatos inadmissíveis são ignorados.

    Parâmetros
    ----------
    t, x : array
        Instantes e preços da série.
    candidatos : lista de sequências (a1, b1, a2, b2)
        Configurações candidatas a comparar.
    refinar : bool
        Se True, cada candidato é também avaliado após refinamento local
        (ver _refinar_localmente).
    gamma : float
        Escala do simplex inicial do refinamento.
    largura_min : float
        Largura mínima admissível da banda M(t) - m(t).
    verbose : bool
        Se True, imprime a log-verosimilhança de cada candidato e, no
        final, a solução escolhida.

    Devolve
    -------
    (melhor_theta, melhor_ll):
        melhor_theta : list [a1, b1, a2, b2], ou None
            O candidato de maior log-verosimilhança entre os admissíveis
            None se nenhum candidato for admissível.
        melhor_ll : float
            A log-verosimilhança de melhor_theta. -inf se nenhum
            candidato for admissível.
    '''
    T = t[-1]
    melhor_theta, melhor_ll = None, -np.inf
    for theta in candidatos:
        if _suporte_minimo(theta, T) <= 0 or _largura_minima(theta, T) < largura_min:
            continue
        ll = classificar_e_estimar(t, x, theta)[2]
        if ll > melhor_ll:
            melhor_ll, melhor_theta = ll, list(theta)
        if refinar:
            theta_ref, ll_ref = _refinar_localmente(t, x, theta, gamma, largura_min)
            if theta_ref is not None and ll_ref > melhor_ll:
                melhor_ll, melhor_theta = ll_ref, list(theta_ref)
        if verbose:
            print(f"  candidato {np.round(theta, 3)} -> LL={ll:.3f}")
    if verbose:
        print(f"\nMelhor solução: LL={melhor_ll:.3f}  theta={np.round(melhor_theta, 4)}")
    return melhor_theta, melhor_ll


def analisar_foco(grelha1, grelha2, grelha_ll, parametros, lik_otimo=None,
                  delta_ll=2.0, verbose=True):
    '''Caracteriza o máximo (foco) de um mapa de identificabilidade.

    A região de tolerância é a componente de {LL >= max(LL) - delta_ll} que
    contém o foco. A sua dimensão indica quão bem determinados estão os dois
    parâmetros. Se lik_otimo for indicado, compara-o com o foco (quando
    verbose=True), o que permite detetar que o otimizador não atingiu o
    máximo naquela direção.

    Parâmetros
    ----------
    grelha1, grelha2, grelha_ll : array
        Saída de mapa_identificabilidade para o mesmo par de parâmetros.
    parametros : tuple de 2 str
        Nomes dos dois parâmetros do mapa.
    lik_otimo : float, opcional
        Log-verosimilhança da solução de referência, para comparação com o foco.
    delta_ll : float
        Queda máxima de log-verosimilhança, face ao máximo do mapa, que
        define a região de tolerância.
    verbose : bool
        Se True, imprime o foco, a região de tolerância e a comparação com
        lik_otimo.

    Devolve
    -------
    foco : dict
        'LL' : float
            Log-verosimilhança do foco (máximo da grelha).
        <nome do 1.º parâmetro>, <nome do 2.º parâmetro> : float
            Coordenadas do foco (chaves com os nomes dados em `parametros`).
        'frac_tolerancia' : float
            Fração das células admissíveis da grelha ocupada pela
            região de tolerância. NaN se não houver células admissíveis.
        'extensao_<parâmetro>' : float
            Extensão (máximo - mínimo) da região de tolerância em cada
            parâmetro, na escala original.
    '''
    p1, p2 = parametros
    idx_max = np.unravel_index(np.nanargmax(grelha_ll), grelha_ll.shape)
    ll_max = float(grelha_ll[idx_max])
    foco = {'LL': ll_max, p1: float(grelha1[idx_max]), p2: float(grelha2[idx_max])}

    acima_limiar = np.isfinite(grelha_ll) & (grelha_ll >= ll_max - delta_ll)
    componentes, _ = label(acima_limiar)
    regiao = (componentes == componentes[idx_max])

    n_admissiveis = int(np.sum(np.isfinite(grelha_ll)))
    foco['frac_tolerancia'] = float(np.sum(regiao) / n_admissiveis) if n_admissiveis else np.nan
    foco[f'extensao_{p1}'] = float(np.ptp(grelha1[regiao]))
    foco[f'extensao_{p2}'] = float(np.ptp(grelha2[regiao]))

    if verbose:
        print(f"Foco (máximo do mapa): LL={foco['LL']:.3f}  "
              f"{p1}={foco[p1]:.3f}  {p2}={foco[p2]:.3f}")
        print(f"  região de tolerância (ΔLL ≤ {delta_ll:g}): "
              f"{foco['frac_tolerancia']:.1%} da grelha admissível")
        print(f"    extensão em {p1}: {foco[f'extensao_{p1}']:.3f}"
              f"   |  extensão em {p2}: {foco[f'extensao_{p2}']:.3f}")
        if lik_otimo is not None:
            print(f"\nÓtimo da estimação: LL={lik_otimo:.3f}  "
                  f"(diferença para o foco: {lik_otimo - foco['LL']:+.3f})")
            if lik_otimo >= foco['LL'] - 1e-6:
                print("  => o ótimo é pelo menos tão bom como o melhor ponto do mapa.")
            else:
                print("  => ATENÇÃO: o mapa tem um ponto melhor do que o ótimo.")
    return foco

## 9. Validação fora da amostra

In [76]:
def instante_cruzamento(theta):
    '''Instante t* em que M(t) = m(t), isto é, t* = (b1 - b2) / (a2 - a1).

    Se 0 < t* <= T, a banda M(t) - m(t) fecha-se dentro do horizonte T. O valor
    pode ser negativo (retas que já se cruzaram e divergem para t > 0). Devolve
    +inf se as retas forem exatamente paralelas (a1 == a2).

    Parâmetros
    ----------
    theta : sequência (a1, b1, a2, b2)
        Declives e ordenadas na origem de M(t) e m(t).

    Devolve
    -------
    float : instante de cruzamento t*, ou +inf se a1 == a2.
    '''
    a1, b1, a2, b2 = theta
    if a1 == a2:
        return np.inf
    return (b1 - b2) / (a2 - a1)


def horizontes_admissiveis(theta, t_teste, fracoes):
    '''Frações do período de teste em que o modelo está definido do início ao fim.

    O modelo só está bem posto enquanto g(t) = M(t) - m(t) > 0. Sendo g afim, é
    positiva num intervalo se e só se o for nos dois extremos. O horizonte com k dias
    é, assim, admissível se g(t_teste[0]) > 0 e g(t_teste[k-1]) > 0. No teste
    exige-se apenas g > 0, e não a largura mínima imposta na estimação.

    Parâmetros
    ----------
    theta : sequência (a1, b1, a2, b2)
        Limiares estimados no treino.
    t_teste : array
        Instantes do período de teste.
    fracoes : sequência de float
        Frações do período de teste a avaliar.

    Devolve
    -------
    (fracoes_ok, t_estrela, n_dias_max):
        fracoes_ok : list de float
            As frações admissíveis, pela ordem dada em `fracoes`.
        t_estrela : float
            Instante de cruzamento das fronteiras (ver instante_cruzamento);
            pode ser negativo ou +inf.
        n_dias_max : int
            Número de dias a simular, o da maior fração admissível. 0 se
            nenhuma fração for admissível.
    '''
    a1, b1, a2, b2 = theta
    t_estrela = instante_cruzamento(theta)
    n_teste = len(t_teste)

    def _n_dias(fracao):
        '''Número de dias de teste correspondente a uma fração do período de teste.
        Uso interno de validacao_multi_horizonte.

        Parâmetros
        ----------
        fracao : float
            Fração do período de teste (0 a 1).

        Devolve
        -------
        int : número de dias, com um mínimo de 1 (t_teste capturado no âmbito
        envolvente).
        '''
        return max(1, int(n_teste * fracao))

    def _largura(t):
        '''Largura da banda, g(t) = M(t) - m(t), no instante t. Uso interno de
        horizontes_admissiveis.

        Parâmetros
        ----------
        t : float
            Instante em que se avalia a largura da banda.

        Devolve
        -------
        float : M(t) - m(t) (theta capturado no âmbito envolvente).
        '''
        return (a1 - a2) * t + (b1 - b2)

    fracoes_ok = [f for f in fracoes
                  if _largura(t_teste[0]) > 0 and _largura(t_teste[_n_dias(f) - 1]) > 0]
    n_dias_max = max((_n_dias(f) for f in fracoes_ok), default=0)
    return fracoes_ok, t_estrela, n_dias_max


def _simular_continuacao(t_futuro, x0, theta, params_regime, regime_inicial, seed=None):
    '''Simula uma trajetória nos instantes t_futuro, a partir de x0 e do regime
    indicado, com comutação nos limiares theta.

    Em cada passo i, o regime é atualizado com a mesma regra do simulador: no
    regime 1 passa-se ao regime 2 se x[i-1] >= M(t[i-1]); no regime 2 passa-se
    ao regime 1 se x[i-1] <= m(t[i-1]).

    Parâmetros
    ----------
    t_futuro : array
        Instantes a simular, incluindo o ponto de partida em t_futuro[0].
    x0 : float
        Preço no ponto de partida.
    theta : sequência (a1, b1, a2, b2)
        Limiares ativos durante a simulação.
    params_regime : dict
        {1: (mu1, sigma1), 2: (mu2, sigma2)} — drift e volatilidade de cada regime.
    regime_inicial : int
        Regime ativo em t_futuro[0].
    seed : int, opcional
        Semente de um gerador independente.

    Devolve
    -------
    array : preços simulados, um por instante de t_futuro (o primeiro é x0).
    '''
    rng = np.random.default_rng(seed)
    a1, b1, a2, b2 = theta
    n = len(t_futuro)
    x = np.empty(n)
    x[0] = x0
    regime = regime_inicial
    for i in range(1, n):
        limiar_sup = a1 * t_futuro[i-1] + b1
        limiar_inf = a2 * t_futuro[i-1] + b2
        if regime == 1 and x[i-1] >= limiar_sup:
            regime = 2
        elif regime == 2 and x[i-1] <= limiar_inf:
            regime = 1
        mu, sigma = params_regime[regime]
        dt = t_futuro[i] - t_futuro[i-1]
        x[i] = x[i-1] * np.exp((mu - 0.5 * sigma**2) * dt
                               + sigma * np.sqrt(dt) * rng.standard_normal())
    return x


def _previsao_central(t_futuro, x0, theta, params_regime, regime_inicial):
    '''Previsão central por troços.

    Propaga x[i] = x[i-1] * exp(mu_r * dt), isto é, o valor esperado do MBG
    dentro de cada regime e aplica a regra de comutação à própria curva.
    A curva resultante não é o valor esperado do processo com comutação,
    mas uma trajetória de referência sem ruído, usada no cálculo do erro
    absoluto e quadrático cumulativos.

    Parâmetros
    ----------
    t_futuro : array
        Instantes a prever, incluindo o ponto de partida em t_futuro[0].
    x0 : float
        Preço no ponto de partida.
    theta : sequência (a1, b1, a2, b2)
        Limiares ativos durante a previsão.
    params_regime : dict
        {1: (mu1, sigma1), 2: (mu2, sigma2)} — drift e volatilidade de cada
        regime.
    regime_inicial : int
        Regime ativo em t_futuro[0], antes de aplicar a regra de comutação.

    Devolve
    -------
    array : previsão central, um valor por instante de t_futuro.
    '''
    a1, b1, a2, b2 = theta
    n = len(t_futuro)
    x = np.empty(n)
    x[0] = x0
    regime = regime_inicial
    for i in range(1, n):
        limiar_sup = a1 * t_futuro[i-1] + b1
        limiar_inf = a2 * t_futuro[i-1] + b2
        if regime == 1 and x[i-1] >= limiar_sup:
            regime = 2
        elif regime == 2 and x[i-1] <= limiar_inf:
            regime = 1
        mu, _ = params_regime[regime]
        dt = t_futuro[i] - t_futuro[i-1]
        x[i] = x[i-1] * np.exp(mu * dt)
    return x


def _metricas_oos(simulacoes, central, x_real, nivel):
    '''Métricas de previsão fora da amostra.

    simulacoes tem dimensão (n_sim, n); central e x_real têm dimensão n. O
    índice 0 é o último ponto do treino (ponto de partida) e é excluído das
    métricas. A banda de previsão é pontual: os quantis são calculados
    separadamente em cada instante, sobre as n_sim trajetórias.

    Métricas:
      - cobertura: fração dos valores reais dentro da banda fechada
        [q_{alpha/2}, q_{1-alpha/2}];
      - largura_media: largura média da banda;
      - err_abs_cum (EAC): soma de |real - central|;
      - err_quad_cum (EQC): soma de (real - central)^2;
      - interval_score: média do interval score de Gneiting & Raftery (2007),
        que soma à largura uma penalização (2/alpha) pela distância a que o valor
        real fica fora da banda.

    Parâmetros
    ----------
    simulacoes : array (n_sim, n)
        Trajetórias simuladas da banda de previsão.
    central : array (n,)
        Previsão central por troços (ver _previsao_central).
    x_real : array (n,)
        Preços efetivamente observados no período de teste.
    nivel : float
        Nível de confiança da banda de previsão (ex.: 0.95).

    Devolve
    -------
    dict com:
        'cobertura', 'largura_media', 'err_abs_cum', 'err_quad_cum',
        'interval_score' : float
            As métricas acima, calculadas nos instantes.
        'lo', 'hi' : array (n,)
            Limites inferior e superior da banda, em todos os instantes.
        'central' : array (n,)
            A previsão central recebida.
    '''
    alpha = 1 - nivel
    lo = np.quantile(simulacoes, alpha / 2, axis=0)
    hi = np.quantile(simulacoes, 1 - alpha / 2, axis=0)
    y, inf, sup, cen = x_real[1:], lo[1:], hi[1:], central[1:]
    interval_score = ((sup - inf)
                      + (2 / alpha) * (inf - y) * (y < inf)
                      + (2 / alpha) * (y - sup) * (y > sup))
    return dict(cobertura=float(np.mean((y >= inf) & (y <= sup))),
                largura_media=float(np.mean(sup - inf)),
                err_abs_cum=float(np.sum(np.abs(y - cen))),
                err_quad_cum=float(np.sum((y - cen)**2)),
                interval_score=float(np.mean(interval_score)),
                lo=lo, hi=hi, central=central)


def _params_por_regime(t_treino, x_treino, theta):
    '''Estima (mu_r, sigma_r) no treino com os limiares theta.

    Parâmetros
    ----------
    t_treino, x_treino : array
        Instantes e preços do período de treino.
    theta : sequência (a1, b1, a2, b2)
        Limiares usados para classificar os regimes no treino.

    Devolve
    -------
    (params_regime, regime_final):
        params_regime : dict {1: (mu1, sigma1), 2: (mu2, sigma2)}, ou None
            Estimativas de máxima verosimilhança por regime. None se algum
            regime não tiver observações suficientes.
        regime_final : int
            Regime sob o qual foi gerado o último incremento do treino
            (regimes[-1]), a usar como regime_inicial das projeções.
            Vale 1 por omissão quando params_regime é None.
    '''
    resultados, regimes, _ = classificar_e_estimar(t_treino, x_treino, theta)
    if resultados[1]['mu'] is None or resultados[2]['mu'] is None:
        return None, 1
    params = {r: (resultados[r]['mu'], resultados[r]['sigma']) for r in [1, 2]}
    return params, int(regimes[-1])


def validacao_multi_horizonte(t_treino, x_treino, t_teste, x_teste, theta_afim, theta_fixo,
                              fracoes_teste=(0.33, 0.66, 1.0), n_sim=2000, nivel=0.95,
                              mostrar=True, plotar=True, prefixo_fig=None):
    '''Validação OOS do MBG e dos modelos de limiares constantes e afins em vários
    horizontes (frações iniciais do período de teste).

    Os limiares não são reestimados por horizonte. Os parâmetros (mu_r, sigma_r) são
    estimados no treino com esses limiares. Cada modelo é simulado apenas até
    ao seu último horizonte admissível (ver horizontes_admissiveis). Nos
    restantes horizontes, as métricas ficam a NaN com o estado 'indefinido'.
    Um modelo cujo treino deixa um regime sem observações suficientes também
    não é simulado e aparece com o estado 'regime vazio'.

    O MBG é tratado como um modelo de limiares inatingíveis (+/-1e9) com os
    mesmos (mu, sigma) nos dois regimes, pelo que permanece no regime 1.

    A trajetória k de cada modelo usa a semente k (k = 0, ..., n_sim-1).

    Em cada horizonte, a estrela (★) na coluna 'Modelo vencedor IS' assinala o
    menor interval score entre os modelos aí definidos. A coluna 'Estado' só é
    incluída se algum modelo estiver 'indefinido' ou com 'regime vazio' em algum
    horizonte.

    Parâmetros
    ----------
    t_treino, x_treino : array
        Instantes e preços do período de treino.
    t_teste, x_teste : array
        Instantes e preços do período de teste.
    theta_afim : sequência (a1, b1, a2, b2)
        Limiares afins estimados no treino.
    theta_fixo : sequência (0, M, 0, m) ou None
        Limiares constantes estimados no treino (None se nenhum par foi admissível).
    fracoes_teste : sequência de float
        Frações do período de teste a avaliar.
    n_sim : int
        Número de trajetórias simuladas por modelo, para a banda de previsão.
    nivel : float
        Nível de confiança da banda de previsão.
    mostrar : bool
        Se True, imprime a tabela de métricas.
    plotar : bool
        Se True, desenha o gráfico da validação.
    prefixo_fig : str, opcional
        Se indicado, o gráfico é guardado com este prefixo.

    Devolve
    -------
    (tabela, detalhes):
        tabela : DataFrame
            Uma linha por horizonte e modelo definido, com as colunas Horizonte,
            Dias_Teste, Modelo, cob (cobertura), largura, EAC, EQC, IS (interval
            score), Estado (se aplicável) e 'Modelo vencedor IS'.
        detalhes : list de dict
            Um por horizonte e modelo, com 'Horizonte', 'Dias', 'Modelo',
            'estado', as métricas de _metricas_oos e as curvas 'lo', 'hi' e
            'central'. Nos estados 'indefinido' e 'regime vazio', as métricas
            são NaN e as curvas None.
    '''
    modelos = ['MBG', 'Limiares Constantes', 'Limiares Afins']

    # Parâmetros por regime (estimados no treino)
    _, mu_mbg, sigma2_mbg = _estimar_mbg(t_treino, x_treino)
    theta_mbg = [0, 1e9, 0, -1e9]
    thetas = {'MBG': theta_mbg, 'Limiares Constantes': theta_fixo, 'Limiares Afins': theta_afim}
    params = {'MBG': ({1: (mu_mbg, np.sqrt(sigma2_mbg)), 2: (mu_mbg, np.sqrt(sigma2_mbg))}, 1)}
    for nome in ['Limiares Constantes', 'Limiares Afins']:
        params[nome] = (_params_por_regime(t_treino, x_treino, thetas[nome])
                        if thetas[nome] is not None else (None, 1))

    # Instantes e preços das projeções, a começar no último ponto do treino
    t_proj = np.concatenate(([t_treino[-1]], t_teste))
    x_proj = np.concatenate(([x_treino[-1]], x_teste))

    # Horizontes admissíveis de cada modelo
    dominio = {}
    for nome in modelos:
        if thetas[nome] is None:
            continue
        fracoes_ok, t_estrela, n_dias_max = horizontes_admissiveis(thetas[nome], t_teste, fracoes_teste)
        dominio[nome] = {'fracoes_ok': fracoes_ok, 't_estrela': t_estrela, 'n_dias_max': n_dias_max}

    if mostrar:
        for nome, d in dominio.items():
            omitidos = [f"{int(f*100)}%" for f in fracoes_teste if f not in d['fracoes_ok']]
            if not omitidos:
                continue
            if t_teste[0] <= d['t_estrela'] <= t_teste[-1]:
                motivo = f"limiares cruzam-se em t*={d['t_estrela']:.3f} anos (dentro do teste)"
            else:
                motivo = (f"limiares já cruzados antes do teste (t*={d['t_estrela']:.3f} anos; "
                          f"m(t) >= M(t) no início do teste)")
            print(f"    {nome}: {motivo} — horizontes omitidos: {', '.join(omitidos)}")

    # Simulação até ao último horizonte admissível
    projecoes = {}
    for nome in dominio:
        params_regime, regime_inicial = params[nome]
        n_dias_max = dominio[nome]['n_dias_max']
        if params_regime is None or n_dias_max == 0:
            projecoes[nome] = None
            continue
        t_sim = t_proj[:n_dias_max + 1]
        simulacoes = np.empty((n_sim, len(t_sim)))
        for k in range(n_sim):
            simulacoes[k] = _simular_continuacao(t_sim, x_treino[-1], thetas[nome],
                                                 params_regime, regime_inicial, seed=k)
        projecoes[nome] = {
            'simulacoes': simulacoes,
            'central': _previsao_central(t_sim, x_treino[-1], thetas[nome],
                                         params_regime, regime_inicial)}

    # Métricas por horizonte
    detalhes = []
    for fracao in fracoes_teste:
        n_dias = max(1, int(len(t_teste) * fracao))
        for nome in modelos:
            if nome not in dominio:
                continue
            base = {'Horizonte': f"{int(fracao*100)}%", 'Dias': n_dias, 'Modelo': nome}
            if params[nome][0] is None:
                estado = 'regime vazio'
            elif fracao not in dominio[nome]['fracoes_ok']:
                estado = 'indefinido'
            else:
                estado = ''
            if estado:
                detalhes.append({**base, 'estado': estado,
                                 'cobertura': np.nan, 'largura_media': np.nan,
                                 'err_abs_cum': np.nan, 'err_quad_cum': np.nan,
                                 'interval_score': np.nan,
                                 'lo': None, 'hi': None, 'central': None})
                continue
            p = projecoes[nome]
            metricas = _metricas_oos(p['simulacoes'][:, :n_dias + 1], p['central'][:n_dias + 1],
                                     x_proj[:n_dias + 1], nivel)
            detalhes.append({**base, 'estado': '', **metricas})

    def _arred(valor, casas):
        '''Arredonda um valor, preservando NaN/infinito. Uso interno de
        validacao_multi_horizonte.

        Parâmetros
        ----------
        valor : float
            Valor a arredondar.
        casas : int
            Número de casas decimais.

        Devolve
        -------
        float : valor arredondado, ou NaN se valor não for finito.
        '''
        return round(valor, casas) if np.isfinite(valor) else np.nan

    tabela = pd.DataFrame([{
        'Horizonte': d['Horizonte'],
        'Dias_Teste': d['Dias'],
        'Modelo': d['Modelo'],
        'cob': _arred(d['cobertura'], 3),
        'largura': _arred(d['largura_media'], 3),
        'EAC': _arred(d['err_abs_cum'], 2),
        'EQC': _arred(d['err_quad_cum'], 2),
        'IS': _arred(d['interval_score'], 3),
        'Estado': d['estado'],
    } for d in detalhes])

    if not tabela.empty:
        def _e_minimo(s):
            '''Assinala, por horizonte, o modelo com menor interval score. Uso interno de
            validacao_multi_horizonte.

            Parâmetros
            ----------
            s : Series
                Coluna de interval scores de um horizonte.

            Devolve
            -------
            Series de bool : True nas posições do valor mínimo.
            '''
            return (s == s.min()) & s.notna()
        vencedor = tabela.groupby('Horizonte')['IS'].transform(_e_minimo)
        tabela['Modelo vencedor IS'] = vencedor.map(lambda v: '★' if v else '')
        if not (tabela['Estado'] != '').any():
            tabela = tabela.drop(columns='Estado')

    # Gráfico
    if plotar:
        plt.figure(figsize=(13, 6.5))
        plt.plot(t_treino, x_treino, color='black', lw=1.0, label='Treino')
        plt.plot(t_teste, x_teste, color='crimson', lw=1.3, label='Teste')

        cores = {'MBG': 'green', 'Limiares Constantes': 'darkorange',
                 'Limiares Afins': 'steelblue'}
        t_total = np.concatenate((t_treino, t_teste))

        for nome in modelos:
            if nome not in dominio:
                continue
            cor = cores[nome]

            # Banda e previsão central
            definidos = [d for d in detalhes if d['Modelo'] == nome and not d['estado']]
            if definidos:
                d = max(definidos, key=lambda d: d['Dias'])
                t_banda = t_proj[1:d['Dias'] + 1]
                plt.fill_between(t_banda, d['lo'][1:], d['hi'][1:], color=cor, alpha=0.12)
                plt.plot(t_banda, d['central'][1:], color=cor, ls='--', lw=1.3,
                         label=nome)
            else:
                plt.plot([], [], color=cor, ls='--', lw=1.3, label=f"{nome}: indefinido no teste")

            # Limiares
            theta = thetas[nome]
            if abs(theta[1]) < 1e6:
                a1, b1, a2, b2 = theta
                plt.plot(t_total, a1 * t_total + b1, color=cor, ls=':', lw=1.0, alpha=0.75)
                plt.plot(t_total, a2 * t_total + b2, color=cor, ls=':', lw=1.0, alpha=0.75)

            t_estrela = dominio[nome]['t_estrela']
            if np.isfinite(t_estrela) and t_total[0] < t_estrela < t_total[-1]:
                plt.axvline(t_estrela, color=cor, ls='-', lw=1.0, alpha=0.6,
                            label=f'{nome}: cruzamento t*')

        plt.axvline(t_treino[-1], color='grey', ls=':', lw=1, label='Fim do treino')
        for fracao in fracoes_teste[:-1]:
            n_dias = max(1, int(len(t_teste) * fracao))
            plt.axvline(t_teste[n_dias - 1], color='purple', ls='-.', lw=1.2, alpha=0.7,
                        label=f'Corte {int(fracao*100)}%')

        plt.xlim(float(t_total[0]), float(t_total[-1]))
        precos = np.concatenate((x_treino, x_teste))
        margem = 0.08 * float(np.ptp(precos))
        plt.ylim(float(precos.min()) - margem, float(precos.max()) + margem)

        plt.xlabel('Tempo (anos)'); plt.ylabel('Preço')
        plt.title(f'Validação fora da amostra (banda de {int(nivel*100)}%)')
        handles, labels = plt.gca().get_legend_handles_labels()
        sem_repetidos = dict(zip(labels, handles))
        plt.legend(sem_repetidos.values(), sem_repetidos.keys(), fontsize=12, loc='upper left')
        plt.grid(True, alpha=0.3); plt.tight_layout()
        if prefixo_fig:
            plt.savefig(f"{prefixo_fig}_oos_multi.png", dpi=110, bbox_inches='tight')
        if mostrar:
            plt.show()
        plt.close()

    if mostrar:
        print(f"\nMétricas OOS por horizonte de teste (frações: {fracoes_teste}):")
        print(tabela.to_string(index=False))
        print("\n  IS: interval score; ★ assinala o modelo vencedor em cada horizonte.")
        print("  EAC/EQC: erro absoluto/quadrático cumulativo da previsão central E[X].")
        if 'Estado' in tabela.columns:
            print(" 'indefinido': horizonte além do cruzamento dos limiares (modelo não definido).")
            print(" 'regime vazio': um dos regimes não tem observações suficientes no treino.")

    return tabela, detalhes

## 10. Gráfico dos regimes

In [77]:
def grafico_regimes(t, x, theta, mostrar=True, guardar=None, titulo=None):
    '''Série de preços colorida pelo regime inferido, com M(t) e m(t) sobrepostos.

    O segmento entre t[i] e t[i+1] recebe a cor de regimes[i+1], o regime sob
    o qual o respetivo incremento foi gerado (a mesma convenção de
    _mle_por_regime).

    Parâmetros
    ----------
    t, x : array
        Instantes e preços da série.
    theta : sequência (a1, b1, a2, b2)
        Limiares usados para classificar os regimes.
    mostrar : bool
        Se True, mostra o gráfico.
    guardar : str, opcional
        Caminho do ficheiro onde o gráfico é guardado.
    titulo : str, opcional
        Título do gráfico.

    Devolve
    -------
    dict com:
        'n_transicoes' : int
            Número de mudanças de regime entre posições consecutivas de
            regimes.
        'frac_regime1', 'frac_regime2' : float
            Fração das observações classificadas em cada regime.
    '''
    _, regimes, _ = classificar_e_estimar(t, x, theta)
    a1, b1, a2, b2 = theta
    cores = {1: 'royalblue', 2: 'darkorange'}
    plt.figure(figsize=(12, 6))
    plt.plot(t, a1 * t + b1, 'r--', alpha=0.7, lw=1.5)
    plt.plot(t, a2 * t + b2, 'g--', alpha=0.7, lw=1.5)
    for i in range(len(t) - 1):
        plt.plot(t[i:i+2], x[i:i+2], color=cores[regimes[i+1]], lw=1.1)
    plt.xlabel('Tempo (anos)'); plt.ylabel('Preço')
    plt.title(titulo or 'Série colorida pelo regime inferido')
    plt.legend(handles=[
        Line2D([0], [0], color='royalblue', lw=2, label='Regime 1'),
        Line2D([0], [0], color='darkorange', lw=2, label='Regime 2'),
        Line2D([0], [0], color='red', lw=1.5, ls='--', label='M(t)'),
        Line2D([0], [0], color='green', lw=1.5, ls='--', label='m(t)')], loc='upper left', fontsize=12)
    plt.grid(True, alpha=0.3); plt.tight_layout()
    if guardar:
        plt.savefig(guardar, dpi=110, bbox_inches='tight')
    if mostrar:
        plt.show()
    plt.close()
    return {'n_transicoes': int(np.sum(regimes[1:] != regimes[:-1])),
            'frac_regime1': float(np.mean(regimes == 1)),
            'frac_regime2': float(np.mean(regimes == 2))}

## 11. Pipeline completo

In [79]:
def _descarregar_precos(ticker, inicio, fim):
    '''Preços de fecho ajustados do Yahoo Finance em [inicio, fim).

    Requer ligação à internet. Com auto_adjust=True, 'Close' já é o preço de
    fecho ajustado.

    Parâmetros
    ----------
    ticker : str
        Símbolo do ativo (convenção Yahoo Finance).
    inicio, fim : str
        Datas de início (inclusive) e fim (exclusive), no formato "AAAA-MM-DD".

    Devolve
    -------
    DataFrame do yfinance, indexado por data, com as colunas Open, High, Low,
    Close e Volume.
    '''
    import yfinance as yf
    return yf.download(ticker, start=inicio, end=fim, auto_adjust=True, progress=False)


def analisar_serie(ticker, inicio, fim, fim_teste=None, fracoes_teste=(0.33, 0.66, 1.0),
                   kappa_min=3,
                   w_list=(10, 21, 63, 126, 252, 504),
                   alpha_list=(0.05, 0.1), delta_list=(-10, -5, 0, 5, 10),
                   p_sup_list=(90, 95, 99), p_inf_list=(10, 5, 1),
                   n_std_list=(1, 1.5, 2), gamma=0.1,
                   fatias=(('a1', 'b1'), ('a2', 'b2'), ('a1', 'a2'), ('b1', 'b2')),
                   n_grid_id=40, n_sim_oos=2000, adotar_melhor_foco=True,
                   nivel_oos=0.95, mostrar=True, prefixo_fig=None):
    '''Análise completa de um ativo.
    
       Passos:
      1. descarrega o treino [inicio, fim) e, se fim_teste for indicado, o teste
         [fim, fim_teste);
      2. estima os limiares afins no treino (sementes + Nelder-Mead);
      3. constrói os mapas de identificabilidade para cada par em `fatias` e,
         se adotar_melhor_foco=True, compara o ótimo inicial com um candidato
         por fatia, cada um avaliado tal como está e após refinamento local; 
         adota o melhor se superar o ótimo inicial em mais;
      4. compara MBG, limiares constantes e limiares afins por AIC/BIC;
      5. diagnostica os parâmetros por regime;
      6. se houver período de teste, faz a validação OOS nas frações indicadas;
      7. desenha a série colorida por regime.

    Parâmetros
    ----------
    ticker : str
        Símbolo do ativo (convenção Yahoo Finance).
    inicio, fim : str
        Início e fim do período de treino, no formato "AAAA-MM-DD".
    fim_teste : str, opcional
        Fim do período de teste. Se None, não há validação OOS.
    fracoes_teste : sequência de float
        Frações do período de teste avaliadas na validação OOS.
    kappa_min : float
        Largura mínima da banda, em movimentos diários típicos do preço.
    w_list, alpha_list, delta_list, p_sup_list, p_inf_list, n_std_list :
        hiperparâmetros das estratégias de geração de sementes.
    gamma : float
        Escala do simplex inicial do Nelder-Mead.
    fatias : sequência de tuplos de 2 str
        Pares de parâmetros para os mapas de identificabilidade.
    n_grid_id : int
        Resolução dos mapas de identificabilidade.
    n_sim_oos : int
        Número de simulações da validação OOS.
    adotar_melhor_foco : bool
        Se True, adota o foco de maior verosimilhança encontrado nos mapas de
        identificabilidade, caso supere a estimativa inicial.
    nivel_oos : float
        Nível de confiança da banda de previsão OOS.
    mostrar : bool
        Se True, imprime as tabelas e mostra os gráficos de cada passo.
    prefixo_fig : str, opcional
        Se indicado, as figuras são guardadas com este prefixo.

    Devolve
    -------
    R : dict com:
        'ticker', 'n_obs', 't', 'x' : identificação e série de treino.
        'theta_afim', 'lik_afim' : estimativa inicial dos limiares afins e a sua
            log-verosimilhança.
        'identificabilidade' : dict {"<p1>_<p2>": {'foco': ...}} por fatia.
        'theta_final', 'lik_final' : limiares após o passo 3 (iguais aos
            iniciais se nenhum foco os superar).
        'comparacao' : DataFrame AIC/BIC (None se comparar_modelos falhar).
        'theta_fixo' : limiares constantes (None se não houver par admissível).
        'diagnostico', 'testes_regimes' : tabela e testes por regime.
        'oos_multi' : (tabela, detalhes) da validação OOS. Só existe se
            houver período de teste.
        'regimes' : dict devolvido por grafico_regimes.
    Se o treino tiver menos de 50 observações, devolve apenas
    {'erro': 'série curta', 'ticker': ticker}.
    '''
    R = {'ticker': ticker}
    separador = '=' * 64
    if fim_teste is not None:
        print(f"{separador}\n  ANÁLISE DE {ticker}\n  Treino: {inicio} a {fim}\n"
              f"  Teste : até {fim_teste}\n{separador}")
    else:
        print(f"{separador}\n  ANÁLISE DE {ticker}  ({inicio} a {fim})\n{separador}")

    # Dados
    dados_treino = _descarregar_precos(ticker, inicio, fim)
    x = dados_treino['Close'].dropna().values.flatten()
    if len(x) < 50:
        print(f"  série de treino demasiado curta ({len(x)} observações) — análise interrompida")
        return {'erro': 'série curta', 'ticker': ticker}
    t = np.arange(len(x)) / DIAS_UTEIS_ANO
    largura_min = kappa_min * _variacao_diaria_tipica(x)
    R.update({'n_obs': len(x), 't': t, 'x': x})
    print(f"\n[1] Treino: {len(x)} observações, {t[-1]:.2f} anos.")

    x_teste, t_teste = np.array([]), np.array([])
    if fim_teste is not None:
        dados_teste = _descarregar_precos(ticker, fim, fim_teste)
        if not dados_teste.empty and not dados_treino.empty:
            dados_teste = dados_teste[dados_teste.index > dados_treino.index[-1]]
        x_teste = dados_teste['Close'].dropna().values.flatten()
        if len(x_teste) > 0:
            t_teste = t[-1] + np.arange(1, len(x_teste) + 1) / DIAS_UTEIS_ANO
            print(f"    Teste: {len(x_teste)} observações até {fim_teste}.")
        else:
            print(f"    AVISO: sem dados entre {fim} e {fim_teste}.")

    # Limiares afins
    print("\n[2] Estimação dos limiares afins no treino (multi-start):")
    sementes = gerar_todas_sementes(t, x, list(w_list), list(alpha_list), list(delta_list),
                                    list(p_sup_list), list(p_inf_list), list(n_std_list))
    definir_espaco_procura(sementes)
    theta, lik, _ = estimar_limiares_afins(t, x, sementes, gamma=gamma, kappa_min=kappa_min)
    R['theta_afim'] = theta
    R['lik_afim'] = lik
    print(f"    theta = {np.round(theta, 4)}   LogLik = {lik:.3f}")

    # Identificabilidade e adoção do melhor foco
    print("\n[3] Identificabilidade (mapas e focos):")
    R['identificabilidade'] = {}
    candidatos = [list(theta)]
    for p1, p2 in fatias:
        ficheiro = f"{prefixo_fig}_id_{p1}{p2}.png" if prefixo_fig else None
        g1, g2, g_ll = mapa_identificabilidade(t, x, theta, parametros=(p1, p2),
                                               n_grid=n_grid_id, largura_min=largura_min,
                                               mostrar=mostrar, guardar=ficheiro)
        foco = analisar_foco(g1, g2, g_ll, (p1, p2), lik_otimo=lik)
        R['identificabilidade'][f"{p1}_{p2}"] = {'foco': foco}
        candidato = list(theta)
        candidato[_INDICE_PARAMETRO[p1]] = foco[p1]
        candidato[_INDICE_PARAMETRO[p2]] = foco[p2]
        candidatos.append(candidato)

    if adotar_melhor_foco:
        theta_cand, lik_cand = selecionar_melhor_candidato(
            t, x, candidatos, gamma=gamma, largura_min=largura_min, refinar=True, verbose=False)
        if lik_cand > lik + 1e-6:
            print(f"    -> foco com maior verosimilhança: LogLik {lik_cand:.3f} > {lik:.3f}; adotado.")
            theta, lik = theta_cand, lik_cand
        else:
            print("    -> ótimo confirmado (nenhum foco o supera).")
    R['theta_final'] = theta
    R['lik_final'] = lik

    # Comparação de modelos
    print("\n[4] Comparação de modelos AIC/BIC:")
    theta_fixo = None
    try:
        tabela_comp, theta_fixo = comparar_modelos(t, x, theta_afim=theta, lik_afim=lik,
                                                   kappa_min=kappa_min)
        R['comparacao'] = tabela_comp
        if mostrar:
            print(tabela_comp.to_string(index=False))
    except Exception as erro:
        print(f"    (comparar_modelos falhou: {erro})")
        R['comparacao'] = None
    R['theta_fixo'] = theta_fixo

    # Diagnóstico por regime
    print("\n[5] Diagnóstico dos parâmetros por regime:")
    tabela_diag, testes = diagnosticar_regimes(t, x, theta, verbose=False)
    R['diagnostico'] = tabela_diag
    R['testes_regimes'] = testes
    if mostrar:
        print(tabela_diag.to_string(index=False))
        if testes:
            print(f"\n    Drift: mu1-mu2={testes['mu1 - mu2']:+.4f} (p={testes['p_mu']}, "
                  f"IC sobrepõem-se={testes['IC_mu_sobrepoe']}) | "
                  f"Vol: s1/s2={testes['sigma1 / sigma2']} (Levene p={testes['p_Levene']}, "
                  f"IC sobrepõem-se={testes['IC_sigma_sobrepoe']})")
            print(f"    Classificação: {testes['rotulo'].upper()}")

    # Validação OOS
    if len(x_teste) > 0:
        print("\n[6] Validação fora da amostra (vários horizontes):")
        R['oos_multi'] = validacao_multi_horizonte(
            t_treino=t, x_treino=x, t_teste=t_teste, x_teste=x_teste,
            theta_afim=theta, theta_fixo=theta_fixo,
            fracoes_teste=fracoes_teste, n_sim=n_sim_oos, nivel=nivel_oos,
            mostrar=mostrar, plotar=mostrar, prefixo_fig=prefixo_fig)
    else:
        print("\n[6] Validação fora da amostra: sem período de teste.")

    # Gráfico dos regimes
    print("\n[7] Série do treino colorida por regime.")
    ficheiro = f"{prefixo_fig}_regimes.png" if prefixo_fig else None
    R['regimes'] = grafico_regimes(t, x, theta, mostrar=mostrar, guardar=ficheiro,
                                   titulo=f'{ticker} — série e regimes inferidos')

    print(f"\n{separador}\n  FIM — {ticker}\n{separador}")
    return R

## Aplicação: ativos e janelas de treino/teste

In [ ]:
ATIVOS = [
    'GILD', 'LLY', 'ABT', 'AMGN', 'MRK',
    'OXY', 'XOM', 'DVN', 'COP', 'HAL',
    'SCHW', 'JPM', 'BAC', 'WFC', 'MS',
    'TGT', 'MCD', 'F', 'TJX', 'AMZN', 
    'CSCO', 'ADBE', 'AAPL', 'NVDA', 'INTC'
]

# Dez janelas de treino/teste de anos inteiros, notação T_{duração,ano de início}
JANELAS = {
    'T1_2021': ('2021-01-01', '2021-12-31', '2022-12-31'),
    'T1_2022': ('2022-01-01', '2022-12-31', '2023-12-31'),
    'T1_2023': ('2023-01-01', '2023-12-31', '2024-12-31'),
    'T1_2024': ('2024-01-01', '2024-12-31', '2025-12-31'),
    'T2_2021': ('2021-01-01', '2022-12-31', '2023-12-31'),
    'T2_2022': ('2022-01-01', '2023-12-31', '2024-12-31'),
    'T2_2023': ('2023-01-01', '2024-12-31', '2025-12-31'),
    'T3_2021': ('2021-01-01', '2023-12-31', '2024-12-31'),
    'T3_2022': ('2022-01-01', '2024-12-31', '2025-12-31'),
    'T4_2021': ('2021-01-01', '2024-12-31', '2025-12-31'),
}

resultados = {}
for ticker in ATIVOS:
    for nome_janela, (inicio, fim, fim_teste) in JANELAS.items():
        resultados[(ticker, nome_janela)] = analisar_serie(
            ticker, inicio, fim, fim_teste=fim_teste,
            fracoes_teste=(0.33, 0.66, 1.0), mostrar=True)

### Critérios de seleção das "melhores" combinações

Uma combinação ativo/janela é considerada um bom resultado quando, simultaneamente:

1. o modelo de **limiares afins vence em AIC e em BIC** face ao MBG e aos
   limiares fixos;
2. os dois regimes têm **boa separação da amostra** (nenhum regime com menos de
   20% das observações de treino);
3. a classificação qualitativa é **"alternância de tendência"**;
4. o modelo afim tem, em média, **melhor interval score** do que os
   outros modelos nos horizontes admissíveis.

In [ ]:
def _resumir_resultado(ticker, janela, R):
    '''Extrai, de um resultado de analisar_serie, os indicadores usados para
    selecionar as melhores combinações ativo/janela: vitória do modelo afim
    em AIC/BIC, separação da amostra entre regimes, classificação qualitativa
    e desempenho preditivo (interval score médio) fora da amostra.

    Parâmetros
    ----------
    ticker : str
        Símbolo do ativo.
    janela : str
        Nome da janela de treino/teste.
    R : dict
        Resultado devolvido por analisar_serie para este ativo/janela.

    Devolve
    -------
    dict : uma linha de resumo, pronta a agregar num DataFrame, com as chaves:
        'Ticker', 'Janela' : identificação.
        'erro' : só presente (e única chave além das anteriores) se a análise
            falhou.
        'Afim_melhor_AIC', 'Afim_melhor_BIC' : bool, o modelo afim tem o
            menor AIC (BIC) entre os modelos comparados.
        'Delta_AIC', 'Delta_BIC' : float, diferença do afim face ao melhor
            modelo.
        'N_regime1', 'N_regime2', 'Fracao_menor' : observações em cada regime
            e fração do regime menos povoado.
        'Classificacao' : rótulo qualitativo dos regimes.
        'IS_medio_afim' : média do interval score do modelo afim nos horizontes
            definidos.
        'Horizontes_vencidos' : horizontes em que o afim tem o menor interval
            score (★); 'Horizontes_total' : horizontes em que o afim está
            definido.
        'IS_medio_outros' : média do interval score das linhas do MBG e dos
            limiares constantes, em conjunto.
    '''
    linha = {'Ticker': ticker, 'Janela': janela}
    if 'erro' in R:
        linha['erro'] = R['erro']
        return linha

    comp = R.get('comparacao')
    if comp is not None and not comp.empty:
        row_afim = comp[comp['Modelo'] == 'Limiares Afins']
        if not row_afim.empty:
            linha['Afim_melhor_AIC'] = bool(row_afim['Melhor AIC'].iloc[0])
            linha['Afim_melhor_BIC'] = bool(row_afim['Melhor BIC'].iloc[0])
            linha['Delta_AIC'] = float(row_afim['Δ AIC'].iloc[0])
            linha['Delta_BIC'] = float(row_afim['Δ BIC'].iloc[0])

    diag = R.get('diagnostico')
    if diag is not None and not diag.empty and 'N_obs' in diag.columns:
        n_obs = diag['N_obs'].dropna().values
        if len(n_obs) == 2:
            linha['N_regime1'] = int(n_obs[0])
            linha['N_regime2'] = int(n_obs[1])
            linha['Fracao_menor'] = round(min(n_obs) / sum(n_obs), 3)

    testes = R.get('testes_regimes') or {}
    linha['Classificacao'] = testes.get('rotulo', '')

    oos = R.get('oos_multi')
    if oos is not None and not oos[0].empty:
        tabela_oos = oos[0]
        afim = tabela_oos[tabela_oos['Modelo'] == 'Limiares Afins']
        outros = tabela_oos[tabela_oos['Modelo'] != 'Limiares Afins']
        if not afim.empty:
            linha['IS_medio_afim'] = round(afim['IS'].mean(skipna=True), 2)
            linha['Horizontes_vencidos'] = int((afim['Modelo vencedor IS'] == '★').sum())
            linha['Horizontes_total'] = int(afim['IS'].notna().sum())
        if not outros.empty:
            linha['IS_medio_outros'] = round(outros['IS'].mean(skipna=True), 2)
    return linha

# Resumo de todas as combinações ativo/janela (incluindo as que falharam)
resumo = pd.DataFrame([_resumir_resultado(ticker, janela, R) for (ticker, janela), R in resultados.items() ])


df = resumo.copy()
if 'erro' in df.columns:
    df = df[df['erro'].isna()].drop(columns='erro')

    
#for col in ['Afim_melhor_AIC', 'Afim_melhor_BIC']:
#    df[col] = df.get(col, False)
#    df[col] = df[col].fillna(False)
#df['Fracao_menor'] = df.get('Fracao_menor', 0.0).fillna(0.0)
#df['Classificacao'] = df.get('Classificacao', '').fillna('')
#df['IS_medio_afim'] = df.get('IS_medio_afim', np.inf).fillna(np.inf)
#df['IS_medio_outros'] = df.get('IS_medio_outros', -np.inf).fillna(-np.inf)

criterios = (
    df['Afim_melhor_AIC'] & df['Afim_melhor_BIC']
    & (df['Fracao_menor'] >= 0.2)
    & (df['Classificacao'] == 'alternância de tendência')
    & (df['IS_medio_afim'] < df['IS_medio_outros'])
)
melhores = df[criterios].sort_values('IS_medio_afim')

print(f"{int(criterios.sum())} de {len(df)} combinações ativo/janela (sem erro) cumprem todos os critérios.\n")
print("Resumo completo (resumo):")
print(resumo.to_string(index=False))
print("\nMelhores combinações (melhores):")
print(melhores.to_string(index=False) if not melhores.empty else "(nenhuma combinação cumpre todos os critérios)")